In [1]:
# ============================================================
# STAGE 4A — RETRIEVAL CORPUS CONSTRUCTION
# ============================================================

from pathlib import Path
import json
import re
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

PROJECT_ROOT = Path("..").resolve()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"

RETRIEVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CORPUS_PATH = (
    PROCESSED_DIR
    / "obesity_development_core_enriched.parquet"
)

OUTPUT_PATH = (
    RETRIEVAL_DIR
    / "retrieval_chunks.parquet"
)


# ============================================================
# 2. Load frozen Stage-3 corpus
# ============================================================

trials = pd.read_parquet(
    CORPUS_PATH
)


# ============================================================
# 3. Restore nested columns
# ============================================================

LIST_COLUMNS = [
    "phases",
    "conditions",
    "keywords",
    "intervention_names",
    "intervention_types",
    "intervention_descriptions",
    "primary_outcomes",
    "secondary_outcomes",
    "countries",
    "states",
    "cities",
    "canonical_interventions",
    "normalized_programs",
]


def parse_list(x):

    if isinstance(x, list):
        return x

    if isinstance(x, str):

        try:

            parsed = json.loads(x)

            if isinstance(parsed, list):
                return parsed

        except Exception:
            pass

    return []


for col in LIST_COLUMNS:

    if col in trials.columns:

        trials[col] = (
            trials[col]
            .apply(parse_list)
        )


# ============================================================
# 4. Text helpers
# ============================================================

def clean_text(x):

    if x is None:
        return ""

    if isinstance(x, float) and pd.isna(x):
        return ""

    text = str(x)

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def join_list(values):

    if not isinstance(values, list):
        return ""

    cleaned = [
        clean_text(x)
        for x in values
        if clean_text(x)
    ]

    return "; ".join(cleaned)


def format_outcomes(outcomes):

    if not isinstance(outcomes, list):
        return ""

    rows = []

    for outcome in outcomes:

        if isinstance(outcome, dict):

            measure = clean_text(
                outcome.get("measure")
            )

            description = clean_text(
                outcome.get("description")
            )

            time_frame = clean_text(
                outcome.get("time_frame")
            )

            parts = []

            if measure:
                parts.append(
                    f"Measure: {measure}"
                )

            if description:
                parts.append(
                    f"Description: {description}"
                )

            if time_frame:
                parts.append(
                    f"Time frame: {time_frame}"
                )

            if parts:
                rows.append(
                    " | ".join(parts)
                )

        else:

            text = clean_text(
                outcome
            )

            if text:
                rows.append(text)

    return "\n".join(rows)


def add_field(
    sections,
    label,
    value
):

    value = clean_text(value)

    if value:

        sections.append(
            f"{label}: {value}"
        )


# ============================================================
# 5. Chunk builders
#
# We use SEMANTIC FIELD-AWARE chunks:
#
# 1. overview
# 2. outcomes
# 3. population
# 4. design
#
# This avoids arbitrary token splits while keeping
# each retrievable unit focused.
# ============================================================

def build_overview_chunk(row):

    sections = []

    add_field(
        sections,
        "NCT ID",
        row.get("nct_id")
    )

    add_field(
        sections,
        "Company",
        row.get("canonical_company")
    )

    add_field(
        sections,
        "Official title",
        row.get("official_title")
    )

    add_field(
        sections,
        "Brief title",
        row.get("brief_title")
    )

    add_field(
        sections,
        "Programs",
        join_list(
            row.get(
                "normalized_programs",
                []
            )
        )
    )

    add_field(
        sections,
        "Phase",
        join_list(
            row.get(
                "phases",
                []
            )
        )
    )

    add_field(
        sections,
        "Status",
        row.get("overall_status")
    )

    add_field(
        sections,
        "Conditions",
        join_list(
            row.get(
                "conditions",
                []
            )
        )
    )

    add_field(
        sections,
        "Brief summary",
        row.get("brief_summary")
    )

    if (
        "detailed_description"
        in row.index
    ):

        add_field(
            sections,
            "Detailed description",
            row.get(
                "detailed_description"
            )
        )

    return "\n".join(sections)


def build_outcomes_chunk(row):

    sections = []

    add_field(
        sections,
        "NCT ID",
        row.get("nct_id")
    )

    add_field(
        sections,
        "Company",
        row.get("canonical_company")
    )

    add_field(
        sections,
        "Programs",
        join_list(
            row.get(
                "normalized_programs",
                []
            )
        )
    )

    add_field(
        sections,
        "Trial title",
        row.get("brief_title")
    )


    primary = format_outcomes(
        row.get(
            "primary_outcomes",
            []
        )
    )

    secondary = format_outcomes(
        row.get(
            "secondary_outcomes",
            []
        )
    )


    add_field(
        sections,
        "Primary outcomes",
        primary
    )

    add_field(
        sections,
        "Secondary outcomes",
        secondary
    )

    return "\n".join(sections)


def build_population_chunk(row):

    sections = []

    add_field(
        sections,
        "NCT ID",
        row.get("nct_id")
    )

    add_field(
        sections,
        "Company",
        row.get("canonical_company")
    )

    add_field(
        sections,
        "Programs",
        join_list(
            row.get(
                "normalized_programs",
                []
            )
        )
    )

    add_field(
        sections,
        "Trial title",
        row.get("brief_title")
    )

    add_field(
        sections,
        "Conditions",
        join_list(
            row.get(
                "conditions",
                []
            )
        )
    )

    add_field(
        sections,
        "Eligibility criteria",
        row.get(
            "eligibility_criteria"
        )
    )

    add_field(
        sections,
        "Countries",
        join_list(
            row.get(
                "countries",
                []
            )
        )
    )

    return "\n".join(sections)


def build_design_chunk(row):

    sections = []

    add_field(
        sections,
        "NCT ID",
        row.get("nct_id")
    )

    add_field(
        sections,
        "Company",
        row.get("canonical_company")
    )

    add_field(
        sections,
        "Programs",
        join_list(
            row.get(
                "normalized_programs",
                []
            )
        )
    )

    add_field(
        sections,
        "Trial title",
        row.get("brief_title")
    )

    add_field(
        sections,
        "Phase",
        join_list(
            row.get(
                "phases",
                []
            )
        )
    )

    add_field(
        sections,
        "Status",
        row.get("overall_status")
    )

    add_field(
        sections,
        "Enrollment",
        row.get("enrollment")
    )

    add_field(
        sections,
        "Start date",
        row.get("start_date")
    )

    add_field(
        sections,
        "Primary completion date",
        row.get(
            "primary_completion_date"
        )
    )

    add_field(
        sections,
        "Completion date",
        row.get(
            "completion_date"
        )
    )

    add_field(
        sections,
        "Allocation",
        row.get("allocation")
    )

    add_field(
        sections,
        "Intervention model",
        row.get(
            "intervention_model"
        )
    )

    add_field(
        sections,
        "Primary purpose",
        row.get(
            "primary_purpose"
        )
    )

    add_field(
        sections,
        "Masking",
        row.get("masking")
    )

    add_field(
        sections,
        "Interventions",
        join_list(
            row.get(
                "intervention_names",
                []
            )
        )
    )

    return "\n".join(sections)


# ============================================================
# 6. Construct chunk corpus
# ============================================================

chunk_builders = {

    "overview":
        build_overview_chunk,

    "outcomes":
        build_outcomes_chunk,

    "population":
        build_population_chunk,

    "design":
        build_design_chunk,
}


chunk_rows = []


for _, row in trials.iterrows():

    for chunk_type, builder in (
        chunk_builders.items()
    ):

        content = builder(row)

        # Skip effectively empty chunks
        if len(content.strip()) < 50:
            continue

        chunk_rows.append({

            "chunk_id":
                (
                    f"{row['nct_id']}"
                    f"__{chunk_type}"
                ),

            "nct_id":
                row["nct_id"],

            "chunk_type":
                chunk_type,

            "canonical_company":
                row[
                    "canonical_company"
                ],

            "phases":
                json.dumps(
                    row.get(
                        "phases",
                        []
                    )
                ),

            "overall_status":
                row[
                    "overall_status"
                ],

            "normalized_programs":
                json.dumps(
                    row.get(
                        "normalized_programs",
                        []
                    )
                ),

            "brief_title":
                row[
                    "brief_title"
                ],

            "content":
                content,
        })


chunks = pd.DataFrame(
    chunk_rows
)


# ============================================================
# 7. Chunk audit
# ============================================================

chunks[
    "character_count"
] = (
    chunks["content"]
    .str.len()
)

chunks[
    "word_count"
] = (
    chunks["content"]
    .str.split()
    .str.len()
)


print(
    "STAGE 4A — RETRIEVAL CORPUS"
)

print("=" * 100)

print(
    f"Source trials: "
    f"{trials['nct_id'].nunique():,}"
)

print(
    f"Retrieval chunks: "
    f"{len(chunks):,}"
)

print(
    f"Trials represented: "
    f"{chunks['nct_id'].nunique():,}"
)


# ============================================================
# 8. Chunks per trial
# ============================================================

print("\n")
print("=" * 100)
print("CHUNKS PER TRIAL")
print("=" * 100)

print(
    chunks.groupby(
        "nct_id"
    )
    .size()
    .describe()
    .round(2)
    .to_string()
)


# ============================================================
# 9. Chunk-type distribution
# ============================================================

print("\n")
print("=" * 100)
print("CHUNK TYPES")
print("=" * 100)

print(
    chunks[
        "chunk_type"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# 10. Chunk-size distribution
# ============================================================

print("\n")
print("=" * 100)
print("WORD COUNT BY CHUNK TYPE")
print("=" * 100)

chunk_size_summary = (
    chunks.groupby(
        "chunk_type"
    )[
        "word_count"
    ]
    .agg(
        [
            "count",
            "min",
            "median",
            "mean",
            "max",
        ]
    )
)

print(
    chunk_size_summary
    .round(1)
    .to_string()
)


print("\nOverall word-count quantiles:")

print(
    chunks[
        "word_count"
    ]
    .quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .round(0)
    .to_string()
)


# ============================================================
# 11. Integrity checks
# ============================================================

assert (
    chunks[
        "chunk_id"
    ].is_unique
)

assert (
    chunks[
        "nct_id"
    ].nunique()
    ==
    trials[
        "nct_id"
    ].nunique()
)

assert (
    chunks[
        "content"
    ].str.strip()
    .ne("")
    .all()
)


# ============================================================
# 12. Save retrieval corpus
# ============================================================

chunks.to_parquet(
    OUTPUT_PATH,
    index=False
)


print("\n")
print("=" * 100)
print("STAGE 4A COMPLETE")
print("=" * 100)

print(
    f"Retrieval corpus:\n"
    f"{OUTPUT_PATH}"
)

print("\nImportant evaluation rule:")

print(
    "Retrieval operates at chunk level, "
    "but metrics will be calculated at NCT/trial level."
)

print(
    "Multiple retrieved chunks from the same trial "
    "must be collapsed before Recall@K / MRR."
)

print("\nObjects ready:")
print("  trials")
print("  chunks")
print("  chunk_size_summary")

STAGE 4A — RETRIEVAL CORPUS
Source trials: 139
Retrieval chunks: 556
Trials represented: 139


CHUNKS PER TRIAL
count    139.0
mean       4.0
std        0.0
min        4.0
25%        4.0
50%        4.0
75%        4.0
max        4.0


CHUNK TYPES
chunk_type
overview      139
outcomes      139
population    139
design        139


WORD COUNT BY CHUNK TYPE
            count  min  median    mean   max
chunk_type                                  
design        139   51    61.0    61.5    98
outcomes      139   38   720.0  1146.3  5931
overview      139   70   136.0   156.8   368
population    139   64   253.0   278.3   707

Overall word-count quantiles:
0.10      59.0
0.25      73.0
0.50     164.0
0.75     322.0
0.90     979.0
0.95    2000.0
0.99    3623.0


STAGE 4A COMPLETE
Retrieval corpus:
C:\Users\shubh\Desktop\Projects\Copilot\data\retrieval\retrieval_chunks.parquet

Important evaluation rule:
Retrieval operates at chunk level, but metrics will be calculated at NCT/trial level.
Multip

In [2]:
# ============================================================
# STAGE 4A v2 — FIELD-AWARE RETRIEVAL CORPUS
# ============================================================

from pathlib import Path
import json
import re
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

PROJECT_ROOT = Path("..").resolve()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"

RETRIEVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CORPUS_PATH = (
    PROCESSED_DIR
    / "obesity_development_core_enriched.parquet"
)

OUTPUT_PATH = (
    RETRIEVAL_DIR
    / "retrieval_chunks.parquet"
)


# ============================================================
# 2. Load frozen corpus
# ============================================================

trials = pd.read_parquet(
    CORPUS_PATH
)


# ============================================================
# 3. Restore nested columns
# ============================================================

LIST_COLUMNS = [
    "phases",
    "conditions",
    "keywords",
    "intervention_names",
    "intervention_types",
    "intervention_descriptions",
    "primary_outcomes",
    "secondary_outcomes",
    "countries",
    "states",
    "cities",
    "canonical_interventions",
    "normalized_programs",
]


def parse_list(x):

    if isinstance(x, list):
        return x

    if isinstance(x, str):

        try:

            parsed = json.loads(x)

            if isinstance(parsed, list):
                return parsed

        except Exception:
            pass

    return []


for col in LIST_COLUMNS:

    if col in trials.columns:

        trials[col] = (
            trials[col]
            .apply(parse_list)
        )


# ============================================================
# 4. Text helpers
# ============================================================

def clean_text(x):

    if x is None:
        return ""

    if (
        isinstance(x, float)
        and pd.isna(x)
    ):
        return ""

    text = str(x)

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def join_list(values):

    if not isinstance(values, list):
        return ""

    values = [
        clean_text(x)
        for x in values
        if clean_text(x)
    ]

    return "; ".join(values)


def add_field(
    sections,
    label,
    value
):

    value = clean_text(value)

    if value:

        sections.append(
            f"{label}: {value}"
        )


# ============================================================
# 5. Long outcome description splitting
#
# Only individual outcome descriptions are split.
# Metadata is repeated in each part.
# ============================================================

MAX_DESCRIPTION_WORDS = 400
DESCRIPTION_OVERLAP_WORDS = 40


def split_text_words(
    text,
    max_words=MAX_DESCRIPTION_WORDS,
    overlap=DESCRIPTION_OVERLAP_WORDS
):

    text = clean_text(text)

    if not text:
        return [""]

    words = text.split()

    if len(words) <= max_words:
        return [text]


    chunks = []

    start = 0

    while start < len(words):

        end = min(
            start + max_words,
            len(words)
        )

        chunks.append(
            " ".join(
                words[start:end]
            )
        )

        if end == len(words):
            break

        start = end - overlap


    return chunks


# ============================================================
# 6. Shared trial metadata
# ============================================================

def base_sections(row):

    sections = []

    add_field(
        sections,
        "NCT ID",
        row.get("nct_id")
    )

    add_field(
        sections,
        "Company",
        row.get(
            "canonical_company"
        )
    )

    add_field(
        sections,
        "Programs",
        join_list(
            row.get(
                "normalized_programs",
                []
            )
        )
    )

    add_field(
        sections,
        "Trial title",
        row.get("brief_title")
    )

    return sections


# ============================================================
# 7. Overview chunk
# ============================================================

def build_overview_chunk(row):

    sections = base_sections(row)

    add_field(
        sections,
        "Official title",
        row.get("official_title")
    )

    add_field(
        sections,
        "Phase",
        join_list(
            row.get("phases", [])
        )
    )

    add_field(
        sections,
        "Status",
        row.get("overall_status")
    )

    add_field(
        sections,
        "Conditions",
        join_list(
            row.get(
                "conditions",
                []
            )
        )
    )

    add_field(
        sections,
        "Brief summary",
        row.get("brief_summary")
    )

    if (
        "detailed_description"
        in row.index
    ):

        add_field(
            sections,
            "Detailed description",
            row.get(
                "detailed_description"
            )
        )

    return "\n".join(sections)


# ============================================================
# 8. Population chunk
# ============================================================

def build_population_chunk(row):

    sections = base_sections(row)

    add_field(
        sections,
        "Conditions",
        join_list(
            row.get(
                "conditions",
                []
            )
        )
    )

    add_field(
        sections,
        "Eligibility criteria",
        row.get(
            "eligibility_criteria"
        )
    )

    add_field(
        sections,
        "Countries",
        join_list(
            row.get(
                "countries",
                []
            )
        )
    )

    return "\n".join(sections)


# ============================================================
# 9. Design chunk
# ============================================================

def build_design_chunk(row):

    sections = base_sections(row)

    add_field(
        sections,
        "Phase",
        join_list(
            row.get("phases", [])
        )
    )

    add_field(
        sections,
        "Status",
        row.get("overall_status")
    )

    add_field(
        sections,
        "Enrollment",
        row.get("enrollment")
    )

    add_field(
        sections,
        "Start date",
        row.get("start_date")
    )

    add_field(
        sections,
        "Primary completion date",
        row.get(
            "primary_completion_date"
        )
    )

    add_field(
        sections,
        "Completion date",
        row.get(
            "completion_date"
        )
    )

    add_field(
        sections,
        "Allocation",
        row.get("allocation")
    )

    add_field(
        sections,
        "Intervention model",
        row.get(
            "intervention_model"
        )
    )

    add_field(
        sections,
        "Primary purpose",
        row.get(
            "primary_purpose"
        )
    )

    add_field(
        sections,
        "Masking",
        row.get("masking")
    )

    add_field(
        sections,
        "Interventions",
        join_list(
            row.get(
                "intervention_names",
                []
            )
        )
    )

    return "\n".join(sections)


# ============================================================
# 10. Individual outcome chunks
#
# One semantic chunk per outcome.
# Exception:
# very long description -> multiple parts.
# ============================================================

def build_outcome_chunks(
    row,
    outcomes,
    outcome_type
):

    result = []

    if not isinstance(outcomes, list):
        return result


    for outcome_idx, outcome in enumerate(
        outcomes,
        start=1
    ):

        if not isinstance(outcome, dict):
            continue


        measure = clean_text(
            outcome.get("measure")
        )

        description = clean_text(
            outcome.get("description")
        )

        time_frame = clean_text(
            outcome.get("time_frame")
        )


        if not (
            measure
            or description
            or time_frame
        ):
            continue


        description_parts = (
            split_text_words(
                description
            )
        )


        for part_idx, description_part in enumerate(
            description_parts,
            start=1
        ):

            sections = base_sections(row)


            add_field(
                sections,
                "Outcome type",
                outcome_type
            )

            add_field(
                sections,
                "Measure",
                measure
            )

            add_field(
                sections,
                "Time frame",
                time_frame
            )

            add_field(
                sections,
                "Description",
                description_part
            )


            chunk_type = (
                f"{outcome_type}_outcome"
            )


            if len(description_parts) == 1:

                chunk_id = (
                    f"{row['nct_id']}"
                    f"__{chunk_type}"
                    f"_{outcome_idx:02d}"
                )

            else:

                chunk_id = (
                    f"{row['nct_id']}"
                    f"__{chunk_type}"
                    f"_{outcome_idx:02d}"
                    f"__part_{part_idx:02d}"
                )


            result.append({

                "chunk_id":
                    chunk_id,

                "nct_id":
                    row["nct_id"],

                "chunk_type":
                    chunk_type,

                "outcome_type":
                    outcome_type,

                "outcome_index":
                    outcome_idx,

                "outcome_part":
                    part_idx,

                "canonical_company":
                    row[
                        "canonical_company"
                    ],

                "phases":
                    json.dumps(
                        row.get(
                            "phases",
                            []
                        )
                    ),

                "overall_status":
                    row[
                        "overall_status"
                    ],

                "normalized_programs":
                    json.dumps(
                        row.get(
                            "normalized_programs",
                            []
                        )
                    ),

                "brief_title":
                    row[
                        "brief_title"
                    ],

                "content":
                    "\n".join(
                        sections
                    ),
            })


    return result


# ============================================================
# 11. Construct retrieval corpus
# ============================================================

chunk_rows = []


for _, row in trials.iterrows():

    # --------------------------------------------------------
    # Trial-level semantic chunks
    # --------------------------------------------------------

    base_chunks = {

        "overview":
            build_overview_chunk(row),

        "population":
            build_population_chunk(row),

        "design":
            build_design_chunk(row),
    }


    for chunk_type, content in (
        base_chunks.items()
    ):

        if len(
            content.strip()
        ) < 50:

            continue


        chunk_rows.append({

            "chunk_id":
                (
                    f"{row['nct_id']}"
                    f"__{chunk_type}"
                ),

            "nct_id":
                row["nct_id"],

            "chunk_type":
                chunk_type,

            "outcome_type":
                None,

            "outcome_index":
                None,

            "outcome_part":
                None,

            "canonical_company":
                row[
                    "canonical_company"
                ],

            "phases":
                json.dumps(
                    row.get(
                        "phases",
                        []
                    )
                ),

            "overall_status":
                row[
                    "overall_status"
                ],

            "normalized_programs":
                json.dumps(
                    row.get(
                        "normalized_programs",
                        []
                    )
                ),

            "brief_title":
                row[
                    "brief_title"
                ],

            "content":
                content,
        })


    # --------------------------------------------------------
    # Primary outcomes
    # --------------------------------------------------------

    chunk_rows.extend(

        build_outcome_chunks(

            row=row,

            outcomes=row.get(
                "primary_outcomes",
                []
            ),

            outcome_type="primary",
        )
    )


    # --------------------------------------------------------
    # Secondary outcomes
    # --------------------------------------------------------

    chunk_rows.extend(

        build_outcome_chunks(

            row=row,

            outcomes=row.get(
                "secondary_outcomes",
                []
            ),

            outcome_type="secondary",
        )
    )


chunks = pd.DataFrame(
    chunk_rows
)


# ============================================================
# 12. Chunk-size metrics
# ============================================================

chunks[
    "character_count"
] = (
    chunks[
        "content"
    ]
    .str.len()
)


chunks[
    "word_count"
] = (
    chunks[
        "content"
    ]
    .str.split()
    .str.len()
)


# ============================================================
# 13. Integrity checks
# ============================================================

assert (
    chunks[
        "chunk_id"
    ].is_unique
)

assert (
    chunks[
        "nct_id"
    ].nunique()
    ==
    trials[
        "nct_id"
    ].nunique()
)

assert (
    chunks[
        "content"
    ]
    .str.strip()
    .ne("")
    .all()
)


# Every trial should retain the three core chunks
core_chunk_counts = (
    chunks.loc[
        chunks[
            "chunk_type"
        ]
        .isin(
            [
                "overview",
                "population",
                "design",
            ]
        )
    ]
    .groupby(
        "nct_id"
    )
    .size()
)


assert (
    core_chunk_counts == 3
).all()


# ============================================================
# 14. Audit
# ============================================================

print(
    "STAGE 4A v2 — RETRIEVAL CORPUS"
)

print("=" * 100)


print(
    f"Source trials: "
    f"{trials['nct_id'].nunique():,}"
)

print(
    f"Retrieval chunks: "
    f"{len(chunks):,}"
)

print(
    f"Trials represented: "
    f"{chunks['nct_id'].nunique():,}"
)


# ------------------------------------------------------------
# chunks per trial
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("CHUNKS PER TRIAL")
print("=" * 100)

print(
    chunks.groupby(
        "nct_id"
    )
    .size()
    .describe()
    .round(2)
    .to_string()
)


# ------------------------------------------------------------
# chunk types
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("CHUNK TYPES")
print("=" * 100)

print(
    chunks[
        "chunk_type"
    ]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# chunk sizes
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("WORD COUNT BY CHUNK TYPE")
print("=" * 100)


chunk_size_summary = (

    chunks.groupby(
        "chunk_type"
    )[
        "word_count"
    ]

    .agg(
        count="count",
        min="min",
        median="median",
        mean="mean",
        p95=lambda x:
            x.quantile(0.95),
        max="max",
    )
)


print(
    chunk_size_summary
    .round(1)
    .to_string()
)


print(
    "\nOverall word-count quantiles:"
)

print(

    chunks[
        "word_count"
    ]

    .quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )

    .round(0)

    .to_string()
)


# ------------------------------------------------------------
# long chunks
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("LONGEST CHUNKS")
print("=" * 100)


print(

    chunks[
        [
            "chunk_id",
            "chunk_type",
            "word_count",
        ]
    ]

    .sort_values(
        "word_count",
        ascending=False
    )

    .head(10)

    .to_string(
        index=False
    )
)


# ============================================================
# 15. Save
# ============================================================

chunks.to_parquet(
    OUTPUT_PATH,
    index=False
)


print("\n")
print("=" * 100)
print("STAGE 4A v2 COMPLETE")
print("=" * 100)

print(
    f"Retrieval corpus:\n"
    f"{OUTPUT_PATH}"
)

print(
    "\nRetrieval unit: chunk"
)

print(
    "Evaluation unit: unique NCT/trial"
)

print(
    "Retrieved chunks from the same NCT "
    "will be collapsed before Recall@K / MRR."
)

print("\nObjects ready:")
print("  trials")
print("  chunks")
print("  chunk_size_summary")

STAGE 4A v2 — RETRIEVAL CORPUS
Source trials: 139
Retrieval chunks: 3,495
Trials represented: 139


CHUNKS PER TRIAL
count    139.00
mean      25.14
std       16.37
min        4.00
25%       14.00
50%       23.00
75%       32.00
max      109.00


CHUNK TYPES
chunk_type
secondary_outcome    2869
primary_outcome       209
overview              139
population            139
design                139


WORD COUNT BY CHUNK TYPE
                   count  min  median   mean    p95  max
chunk_type                                              
design               139   51    61.0   61.5   73.0   98
overview             139   70   136.0  156.8  303.4  368
population           139   64   253.0  278.3  498.5  707
primary_outcome      209   31    62.0   79.6  161.4  211
secondary_outcome   2869   30    65.0   79.6  165.0  232

Overall word-count quantiles:
0.10     47.0
0.25     55.0
0.50     67.0
0.75    104.0
0.90    156.0
0.95    193.0
0.99    332.0


LONGEST CHUNKS
               chunk_id chun

In [3]:
# ============================================================
# STAGE 4A v3 — FIELD-AWARE RETRIEVAL CORPUS
# Secondary outcomes packed to reduce document multiplicity bias
# ============================================================

from pathlib import Path
import json
import re
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

PROJECT_ROOT = Path("..").resolve()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"

RETRIEVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CORPUS_PATH = (
    PROCESSED_DIR
    / "obesity_development_core_enriched.parquet"
)

OUTPUT_PATH = (
    RETRIEVAL_DIR
    / "retrieval_chunks.parquet"
)


# ============================================================
# 2. Load frozen corpus
# ============================================================

trials = pd.read_parquet(
    CORPUS_PATH
)


# ============================================================
# 3. Restore nested columns
# ============================================================

LIST_COLUMNS = [
    "phases",
    "conditions",
    "keywords",
    "intervention_names",
    "intervention_types",
    "intervention_descriptions",
    "primary_outcomes",
    "secondary_outcomes",
    "countries",
    "states",
    "cities",
    "canonical_interventions",
    "normalized_programs",
]


def parse_list(x):

    if isinstance(x, list):
        return x

    if isinstance(x, str):

        try:
            parsed = json.loads(x)

            if isinstance(parsed, list):
                return parsed

        except Exception:
            pass

    return []


for col in LIST_COLUMNS:

    if col in trials.columns:

        trials[col] = (
            trials[col]
            .apply(parse_list)
        )


# ============================================================
# 4. Helpers
# ============================================================

def clean_text(x):

    if x is None:
        return ""

    if (
        isinstance(x, float)
        and pd.isna(x)
    ):
        return ""

    text = str(x)

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def join_list(values):

    if not isinstance(values, list):
        return ""

    values = [
        clean_text(x)
        for x in values
        if clean_text(x)
    ]

    return "; ".join(values)


def add_field(
    sections,
    label,
    value
):

    value = clean_text(value)

    if value:

        sections.append(
            f"{label}: {value}"
        )


def word_count(text):

    return len(
        clean_text(text).split()
    )


# ============================================================
# 5. Long-description splitting
#
# Still protects us against unusually verbose single outcomes.
# ============================================================

MAX_OUTCOME_DESCRIPTION_WORDS = 300
OUTCOME_DESCRIPTION_OVERLAP = 30


def split_text_words(
    text,
    max_words=MAX_OUTCOME_DESCRIPTION_WORDS,
    overlap=OUTCOME_DESCRIPTION_OVERLAP
):

    text = clean_text(text)

    if not text:
        return [""]

    words = text.split()

    if len(words) <= max_words:
        return [text]

    parts = []

    start = 0

    while start < len(words):

        end = min(
            start + max_words,
            len(words)
        )

        parts.append(
            " ".join(
                words[start:end]
            )
        )

        if end == len(words):
            break

        start = end - overlap

    return parts


# ============================================================
# 6. Shared metadata
# ============================================================

def base_sections(row):

    sections = []

    add_field(
        sections,
        "NCT ID",
        row.get("nct_id")
    )

    add_field(
        sections,
        "Company",
        row.get(
            "canonical_company"
        )
    )

    add_field(
        sections,
        "Programs",
        join_list(
            row.get(
                "normalized_programs",
                []
            )
        )
    )

    add_field(
        sections,
        "Trial title",
        row.get("brief_title")
    )

    return sections


# ============================================================
# 7. Core chunks
# ============================================================

def build_overview_chunk(row):

    sections = base_sections(row)

    add_field(
        sections,
        "Official title",
        row.get("official_title")
    )

    add_field(
        sections,
        "Phase",
        join_list(
            row.get("phases", [])
        )
    )

    add_field(
        sections,
        "Status",
        row.get("overall_status")
    )

    add_field(
        sections,
        "Conditions",
        join_list(
            row.get(
                "conditions",
                []
            )
        )
    )

    add_field(
        sections,
        "Brief summary",
        row.get("brief_summary")
    )

    if (
        "detailed_description"
        in row.index
    ):

        add_field(
            sections,
            "Detailed description",
            row.get(
                "detailed_description"
            )
        )

    return "\n".join(sections)


def build_population_chunk(row):

    sections = base_sections(row)

    add_field(
        sections,
        "Conditions",
        join_list(
            row.get(
                "conditions",
                []
            )
        )
    )

    add_field(
        sections,
        "Eligibility criteria",
        row.get(
            "eligibility_criteria"
        )
    )

    add_field(
        sections,
        "Countries",
        join_list(
            row.get(
                "countries",
                []
            )
        )
    )

    return "\n".join(sections)


def build_design_chunk(row):

    sections = base_sections(row)

    add_field(
        sections,
        "Phase",
        join_list(
            row.get("phases", [])
        )
    )

    add_field(
        sections,
        "Status",
        row.get("overall_status")
    )

    add_field(
        sections,
        "Enrollment",
        row.get("enrollment")
    )

    add_field(
        sections,
        "Start date",
        row.get("start_date")
    )

    add_field(
        sections,
        "Primary completion date",
        row.get(
            "primary_completion_date"
        )
    )

    add_field(
        sections,
        "Completion date",
        row.get(
            "completion_date"
        )
    )

    add_field(
        sections,
        "Allocation",
        row.get("allocation")
    )

    add_field(
        sections,
        "Intervention model",
        row.get(
            "intervention_model"
        )
    )

    add_field(
        sections,
        "Primary purpose",
        row.get(
            "primary_purpose"
        )
    )

    add_field(
        sections,
        "Masking",
        row.get("masking")
    )

    add_field(
        sections,
        "Interventions",
        join_list(
            row.get(
                "intervention_names",
                []
            )
        )
    )

    return "\n".join(sections)


# ============================================================
# 8. Primary outcomes
#
# Keep one semantic chunk per primary outcome.
# ============================================================

def build_primary_outcome_chunks(row):

    results = []

    outcomes = row.get(
        "primary_outcomes",
        []
    )

    if not isinstance(outcomes, list):
        return results


    for outcome_idx, outcome in enumerate(
        outcomes,
        start=1
    ):

        if not isinstance(outcome, dict):
            continue


        measure = clean_text(
            outcome.get("measure")
        )

        description = clean_text(
            outcome.get("description")
        )

        time_frame = clean_text(
            outcome.get("time_frame")
        )


        if not (
            measure
            or description
            or time_frame
        ):
            continue


        description_parts = (
            split_text_words(
                description
            )
        )


        for part_idx, description_part in enumerate(
            description_parts,
            start=1
        ):

            sections = base_sections(row)

            add_field(
                sections,
                "Outcome type",
                "Primary"
            )

            add_field(
                sections,
                "Measure",
                measure
            )

            add_field(
                sections,
                "Time frame",
                time_frame
            )

            add_field(
                sections,
                "Description",
                description_part
            )


            chunk_id = (
                f"{row['nct_id']}"
                f"__primary_outcome"
                f"_{outcome_idx:02d}"
            )

            if len(
                description_parts
            ) > 1:

                chunk_id += (
                    f"__part_{part_idx:02d}"
                )


            results.append({

                "chunk_id":
                    chunk_id,

                "nct_id":
                    row["nct_id"],

                "chunk_type":
                    "primary_outcome",

                "outcome_type":
                    "primary",

                "outcome_start_index":
                    outcome_idx,

                "outcome_end_index":
                    outcome_idx,

                "outcome_part":
                    part_idx,

                "canonical_company":
                    row[
                        "canonical_company"
                    ],

                "phases":
                    json.dumps(
                        row.get(
                            "phases",
                            []
                        )
                    ),

                "overall_status":
                    row[
                        "overall_status"
                    ],

                "normalized_programs":
                    json.dumps(
                        row.get(
                            "normalized_programs",
                            []
                        )
                    ),

                "brief_title":
                    row[
                        "brief_title"
                    ],

                "content":
                    "\n".join(
                        sections
                    ),
            })


    return results


# ============================================================
# 9. Secondary outcome formatting
# ============================================================

def build_secondary_outcome_blocks(row):

    """
    Convert secondary outcomes into compact semantic blocks.

    A very long individual outcome may yield multiple blocks,
    but ordinary outcomes yield exactly one.
    """

    blocks = []

    outcomes = row.get(
        "secondary_outcomes",
        []
    )

    if not isinstance(outcomes, list):
        return blocks


    for outcome_idx, outcome in enumerate(
        outcomes,
        start=1
    ):

        if not isinstance(outcome, dict):
            continue


        measure = clean_text(
            outcome.get("measure")
        )

        description = clean_text(
            outcome.get("description")
        )

        time_frame = clean_text(
            outcome.get("time_frame")
        )


        if not (
            measure
            or description
            or time_frame
        ):
            continue


        description_parts = (
            split_text_words(
                description
            )
        )


        for part_idx, description_part in enumerate(
            description_parts,
            start=1
        ):

            sections = [
                (
                    f"Secondary outcome "
                    f"{outcome_idx}"
                )
            ]


            add_field(
                sections,
                "Measure",
                measure
            )

            add_field(
                sections,
                "Time frame",
                time_frame
            )

            add_field(
                sections,
                "Description",
                description_part
            )


            block_text = (
                "\n".join(
                    sections
                )
            )


            blocks.append({

                "outcome_index":
                    outcome_idx,

                "outcome_part":
                    part_idx,

                "text":
                    block_text,

                "word_count":
                    word_count(
                        block_text
                    ),
            })


    return blocks


# ============================================================
# 10. Pack secondary outcomes
#
# Key change from v2:
# several secondary outcomes share one chunk.
# No outcome is dropped.
# ============================================================

SECONDARY_PACK_TARGET_WORDS = 350


def pack_secondary_blocks(blocks):

    if not blocks:
        return []


    packs = []

    current_blocks = []
    current_words = 0


    for block in blocks:

        block_words = (
            block[
                "word_count"
            ]
        )


        # Start a new pack if adding this block
        # would materially exceed the target.
        if (
            current_blocks
            and
            current_words
            + block_words
            > SECONDARY_PACK_TARGET_WORDS
        ):

            packs.append(
                current_blocks
            )

            current_blocks = []
            current_words = 0


        current_blocks.append(
            block
        )

        current_words += (
            block_words
        )


    if current_blocks:

        packs.append(
            current_blocks
        )


    return packs


def build_secondary_outcome_chunks(row):

    blocks = (
        build_secondary_outcome_blocks(
            row
        )
    )

    packs = (
        pack_secondary_blocks(
            blocks
        )
    )

    results = []


    for pack_idx, pack in enumerate(
        packs,
        start=1
    ):

        sections = base_sections(row)


        add_field(
            sections,
            "Outcome type",
            "Secondary"
        )


        sections.append(
            "\n\n".join(
                block["text"]
                for block in pack
            )
        )


        start_idx = min(
            block[
                "outcome_index"
            ]
            for block in pack
        )

        end_idx = max(
            block[
                "outcome_index"
            ]
            for block in pack
        )


        results.append({

            "chunk_id":
                (
                    f"{row['nct_id']}"
                    f"__secondary_outcomes"
                    f"_{pack_idx:02d}"
                ),

            "nct_id":
                row["nct_id"],

            "chunk_type":
                "secondary_outcomes",

            "outcome_type":
                "secondary",

            "outcome_start_index":
                start_idx,

            "outcome_end_index":
                end_idx,

            "outcome_part":
                None,

            "canonical_company":
                row[
                    "canonical_company"
                ],

            "phases":
                json.dumps(
                    row.get(
                        "phases",
                        []
                    )
                ),

            "overall_status":
                row[
                    "overall_status"
                ],

            "normalized_programs":
                json.dumps(
                    row.get(
                        "normalized_programs",
                        []
                    )
                ),

            "brief_title":
                row[
                    "brief_title"
                ],

            "content":
                "\n".join(
                    sections
                ),
        })


    return results


# ============================================================
# 11. Construct retrieval corpus
# ============================================================

chunk_rows = []


for _, row in trials.iterrows():

    # --------------------------------------------------------
    # Core semantic chunks
    # --------------------------------------------------------

    core_chunks = {

        "overview":
            build_overview_chunk(row),

        "population":
            build_population_chunk(row),

        "design":
            build_design_chunk(row),
    }


    for chunk_type, content in (
        core_chunks.items()
    ):

        if len(
            content.strip()
        ) < 50:

            continue


        chunk_rows.append({

            "chunk_id":
                (
                    f"{row['nct_id']}"
                    f"__{chunk_type}"
                ),

            "nct_id":
                row["nct_id"],

            "chunk_type":
                chunk_type,

            "outcome_type":
                None,

            "outcome_start_index":
                None,

            "outcome_end_index":
                None,

            "outcome_part":
                None,

            "canonical_company":
                row[
                    "canonical_company"
                ],

            "phases":
                json.dumps(
                    row.get(
                        "phases",
                        []
                    )
                ),

            "overall_status":
                row[
                    "overall_status"
                ],

            "normalized_programs":
                json.dumps(
                    row.get(
                        "normalized_programs",
                        []
                    )
                ),

            "brief_title":
                row[
                    "brief_title"
                ],

            "content":
                content,
        })


    # --------------------------------------------------------
    # Primary outcomes: one per outcome
    # --------------------------------------------------------

    chunk_rows.extend(
        build_primary_outcome_chunks(
            row
        )
    )


    # --------------------------------------------------------
    # Secondary outcomes: packed
    # --------------------------------------------------------

    chunk_rows.extend(
        build_secondary_outcome_chunks(
            row
        )
    )


chunks = pd.DataFrame(
    chunk_rows
)


# ============================================================
# 12. Size metrics
# ============================================================

chunks[
    "character_count"
] = (
    chunks[
        "content"
    ]
    .str.len()
)


chunks[
    "word_count"
] = (
    chunks[
        "content"
    ]
    .str.split()
    .str.len()
)


# ============================================================
# 13. Integrity checks
# ============================================================

assert (
    chunks[
        "chunk_id"
    ].is_unique
)


assert (
    chunks[
        "nct_id"
    ].nunique()
    ==
    trials[
        "nct_id"
    ].nunique()
)


assert (
    chunks[
        "content"
    ]
    .str.strip()
    .ne("")
    .all()
)


# Every trial should have exactly
# overview + population + design.
core_counts = (

    chunks.loc[
        chunks[
            "chunk_type"
        ]
        .isin(
            [
                "overview",
                "population",
                "design",
            ]
        )
    ]

    .groupby(
        "nct_id"
    )

    .size()
)


assert (
    core_counts == 3
).all()


# ============================================================
# 14. Verify outcome coverage
#
# No primary/secondary outcome may disappear during packing.
# ============================================================

source_primary_count = sum(

    len(x)

    for x in trials[
        "primary_outcomes"
    ]

    if isinstance(x, list)
)


source_secondary_count = sum(

    len(x)

    for x in trials[
        "secondary_outcomes"
    ]

    if isinstance(x, list)
)


indexed_primary_indices = (

    chunks.loc[
        chunks[
            "chunk_type"
        ]
        == "primary_outcome",
        [
            "nct_id",
            "outcome_start_index",
        ]
    ]

    .drop_duplicates()

    .shape[0]
)


secondary_ranges = (

    chunks.loc[
        chunks[
            "chunk_type"
        ]
        == "secondary_outcomes",
        [
            "nct_id",
            "outcome_start_index",
            "outcome_end_index",
        ]
    ]
)


indexed_secondary_count = 0


for _, r in (
    secondary_ranges.iterrows()
):

    indexed_secondary_count += (

        int(
            r[
                "outcome_end_index"
            ]
        )

        -

        int(
            r[
                "outcome_start_index"
            ]
        )

        + 1
    )


assert (
    indexed_primary_indices
    ==
    source_primary_count
), (
    "Primary outcome coverage mismatch"
)


assert (
    indexed_secondary_count
    ==
    source_secondary_count
), (
    "Secondary outcome coverage mismatch"
)


# ============================================================
# 15. Audit
# ============================================================

print(
    "STAGE 4A v3 — RETRIEVAL CORPUS"
)

print("=" * 100)


print(
    f"Source trials: "
    f"{trials['nct_id'].nunique():,}"
)

print(
    f"Retrieval chunks: "
    f"{len(chunks):,}"
)

print(
    f"Trials represented: "
    f"{chunks['nct_id'].nunique():,}"
)


print(
    f"\nSource primary outcomes: "
    f"{source_primary_count:,}"
)

print(
    f"Source secondary outcomes: "
    f"{source_secondary_count:,}"
)

print(
    "Outcome coverage check: PASSED"
)


# ============================================================
# 16. Chunks per trial
# ============================================================

print("\n")
print("=" * 100)
print("CHUNKS PER TRIAL")
print("=" * 100)


chunks_per_trial = (

    chunks.groupby(
        "nct_id"
    )

    .size()
)


print(
    chunks_per_trial
    .describe()
    .round(2)
    .to_string()
)


# ============================================================
# 17. Chunk types
# ============================================================

print("\n")
print("=" * 100)
print("CHUNK TYPES")
print("=" * 100)


print(
    chunks[
        "chunk_type"
    ]
    .value_counts()
    .to_string()
)


# ============================================================
# 18. Word-count audit
# ============================================================

print("\n")
print("=" * 100)
print("WORD COUNT BY CHUNK TYPE")
print("=" * 100)


chunk_size_summary = (

    chunks.groupby(
        "chunk_type"
    )[
        "word_count"
    ]

    .agg(
        count="count",
        min="min",
        median="median",
        mean="mean",
        p95=lambda x:
            x.quantile(0.95),
        max="max",
    )
)


print(
    chunk_size_summary
    .round(1)
    .to_string()
)


print(
    "\nOverall word-count quantiles:"
)


print(

    chunks[
        "word_count"
    ]

    .quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )

    .round(0)

    .to_string()
)


# ============================================================
# 19. Multiplicity audit
# ============================================================

print("\n")
print("=" * 100)
print("CHUNK MULTIPLICITY AUDIT")
print("=" * 100)


print(
    f"Median chunks/trial: "
    f"{chunks_per_trial.median():.0f}"
)

print(
    f"P95 chunks/trial: "
    f"{chunks_per_trial.quantile(0.95):.0f}"
)

print(
    f"Max chunks/trial: "
    f"{chunks_per_trial.max():.0f}"
)


print(
    "\nTrials with most chunks:"
)


print(

    chunks_per_trial
    .sort_values(
        ascending=False
    )
    .head(10)
    .to_string()
)


# ============================================================
# 20. Longest chunks
# ============================================================

print("\n")
print("=" * 100)
print("LONGEST CHUNKS")
print("=" * 100)


print(

    chunks[
        [
            "chunk_id",
            "chunk_type",
            "word_count",
        ]
    ]

    .sort_values(
        "word_count",
        ascending=False
    )

    .head(10)

    .to_string(
        index=False
    )
)


# ============================================================
# 21. Save
# ============================================================

chunks.to_parquet(
    OUTPUT_PATH,
    index=False
)


print("\n")
print("=" * 100)
print("STAGE 4A v3 COMPLETE")
print("=" * 100)


print(
    f"Retrieval corpus:\n"
    f"{OUTPUT_PATH}"
)

print(
    "\nRetrieval unit: chunk"
)

print(
    "Evaluation unit: unique NCT/trial"
)

print(
    "Secondary outcomes are packed rather than "
    "indexed one-per-outcome."
)

print(
    "Retrieved chunks from the same NCT will be "
    "collapsed before Recall@K / MRR."
)

print("\nObjects ready:")
print("  trials")
print("  chunks")
print("  chunks_per_trial")
print("  chunk_size_summary")

STAGE 4A v3 — RETRIEVAL CORPUS
Source trials: 139
Retrieval chunks: 1,164
Trials represented: 139

Source primary outcomes: 209
Source secondary outcomes: 2,869
Outcome coverage check: PASSED


CHUNKS PER TRIAL
count    139.00
mean       8.37
std        3.96
min        4.00
25%        5.00
50%        7.00
75%       10.00
max       24.00


CHUNK TYPES
chunk_type
secondary_outcomes    538
primary_outcome       209
overview              139
population            139
design                139


WORD COUNT BY CHUNK TYPE
                    count  min  median   mean    p95  max
chunk_type                                               
design                139   51    61.0   61.5   73.0   98
overview              139   70   136.0  156.8  303.4  368
population            139   64   253.0  278.3  498.5  707
primary_outcome       209   31    62.0   79.6  161.4  211
secondary_outcomes    538   43   337.0  305.5  379.0  393

Overall word-count quantiles:
0.10     56.0
0.25     78.0
0.50    218.0


In [4]:
# ============================================================
# STAGE 4B — BM25 RETRIEVAL BASELINE
# ============================================================

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi


# ============================================================
# 1. Paths
# ============================================================

PROJECT_ROOT = Path("..").resolve()

RETRIEVAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "retrieval"
)

EVAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "retrieval"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CHUNKS_PATH = (
    RETRIEVAL_DIR
    / "retrieval_chunks.parquet"
)

EVAL_PATH = (
    EVAL_DIR
    / "evaluation_questions_v2.csv"
)


QUESTION_RESULTS_PATH = (
    RESULTS_DIR
    / "bm25_question_metrics.csv"
)

RANKINGS_PATH = (
    RESULTS_DIR
    / "bm25_trial_rankings.parquet"
)

SUMMARY_PATH = (
    RESULTS_DIR
    / "bm25_summary.csv"
)


# ============================================================
# 2. Load frozen retrieval corpus + evaluation set
# ============================================================

chunks = pd.read_parquet(
    CHUNKS_PATH
)

evaluation = pd.read_csv(
    EVAL_PATH
)


# Evaluate ONLY retrieval / hybrid questions
eval_rag = (
    evaluation.loc[
        evaluation[
            "requires_evidence_retrieval"
        ]
        == True
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(eval_rag) == 19, (
    f"Expected 19 retrieval/hybrid questions, "
    f"found {len(eval_rag)}"
)


# ============================================================
# 3. Helpers
# ============================================================

def parse_json_list(x):

    if isinstance(x, list):
        return x

    if isinstance(x, str):

        try:
            value = json.loads(x)

            if isinstance(value, list):
                return value

        except Exception:
            pass

    return []


# Deliberately simple baseline:
# - lowercase
# - retain letters, numbers and hyphenated drug/code terms
# - no stemming
# - no synonym expansion
# - no query rewriting
# - no metadata filtering

TOKEN_PATTERN = re.compile(
    r"[a-z0-9]+(?:-[a-z0-9]+)*"
)


def tokenize(text):

    text = str(text).lower()

    return TOKEN_PATTERN.findall(
        text
    )


# ============================================================
# 4. Tokenize corpus
# ============================================================

tokenized_corpus = (
    chunks[
        "content"
    ]
    .fillna("")
    .apply(tokenize)
    .tolist()
)


print(
    "Building BM25 index..."
)


bm25 = BM25Okapi(
    tokenized_corpus
)


print(
    f"Indexed chunks: {len(chunks):,}"
)


# ============================================================
# 5. Chunk ranking -> trial ranking
#
# Important:
# BM25 scores chunks.
#
# Evaluation is at unique NCT/trial level.
#
# Trial score = maximum chunk score for that trial.
# ============================================================

def rank_trials_bm25(
    query,
    top_trials=20
):

    query_tokens = tokenize(
        query
    )


    scores = bm25.get_scores(
        query_tokens
    )


    ranked_chunks = (
        chunks[
            [
                "chunk_id",
                "nct_id",
                "chunk_type",
                "canonical_company",
                "brief_title",
                "content",
            ]
        ]
        .copy()
    )


    ranked_chunks[
        "bm25_score"
    ] = scores


    ranked_chunks = (
        ranked_chunks
        .sort_values(
            [
                "bm25_score",
                "chunk_id",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Collapse multiple chunks from same trial
    #
    # First occurrence = highest scoring chunk for that NCT
    # --------------------------------------------------------

    ranked_trials = (
        ranked_chunks
        .drop_duplicates(
            subset="nct_id",
            keep="first"
        )
        .reset_index(drop=True)
    )


    ranked_trials[
        "trial_rank"
    ] = (
        np.arange(
            len(ranked_trials)
        )
        + 1
    )


    return (
        ranked_trials
        .head(top_trials)
        .copy()
    )


# ============================================================
# 6. Retrieval metrics
# ============================================================

def recall_at_k(
    ranked_ids,
    gold_ids,
    k
):

    gold = set(
        gold_ids
    )

    if len(gold) == 0:
        return np.nan


    retrieved = set(
        ranked_ids[:k]
    )


    return (
        len(
            retrieved
            & gold
        )
        /
        len(gold)
    )


def reciprocal_rank(
    ranked_ids,
    gold_ids
):

    gold = set(
        gold_ids
    )

    if len(gold) == 0:
        return np.nan


    for rank, nct_id in enumerate(
        ranked_ids,
        start=1
    ):

        if nct_id in gold:

            return (
                1.0 / rank
            )

    return 0.0


def hit_at_k(
    ranked_ids,
    gold_ids,
    k
):

    gold = set(
        gold_ids
    )

    if len(gold) == 0:
        return np.nan


    return float(
        len(
            set(
                ranked_ids[:k]
            )
            & gold
        )
        > 0
    )


# ============================================================
# 7. Run evaluation
# ============================================================

question_rows = []
ranking_rows = []


for _, q in eval_rag.iterrows():

    question_id = q[
        "question_id"
    ]

    query = q[
        "question"
    ]

    gold_ids = parse_json_list(
        q[
            "gold_evidence_nct_ids"
        ]
    )


    assert len(gold_ids) > 0, (
        f"{question_id} has no gold evidence."
    )


    # Rank enough unique trials for:
    # Recall@5, Recall@10 and error analysis
    ranked = rank_trials_bm25(
        query=query,
        top_trials=20
    )


    ranked_ids = (
        ranked[
            "nct_id"
        ]
        .tolist()
    )


    relevant_ranks = [

        rank

        for rank, nct_id
        in enumerate(
            ranked_ids,
            start=1
        )

        if nct_id in set(
            gold_ids
        )
    ]


    first_relevant_rank = (

        min(
            relevant_ranks
        )

        if relevant_ranks

        else None
    )


    question_rows.append({

        "question_id":
            question_id,

        "question_type":
            q[
                "question_type"
            ],

        "route":
            q[
                "route"
            ],

        "question":
            query,

        "gold_count":
            len(
                gold_ids
            ),

        "recall_at_5":
            recall_at_k(
                ranked_ids,
                gold_ids,
                5
            ),

        "recall_at_10":
            recall_at_k(
                ranked_ids,
                gold_ids,
                10
            ),

        "hit_at_5":
            hit_at_k(
                ranked_ids,
                gold_ids,
                5
            ),

        "hit_at_10":
            hit_at_k(
                ranked_ids,
                gold_ids,
                10
            ),

        "mrr":
            reciprocal_rank(
                ranked_ids,
                gold_ids
            ),

        "first_relevant_rank":
            first_relevant_rank,

        "top_1_nct":
            ranked_ids[0]
            if ranked_ids
            else None,

        "top_1_is_gold":
            bool(
                ranked_ids
                and
                ranked_ids[0]
                in set(gold_ids)
            ),
    })


    # --------------------------------------------------------
    # Save top-20 trial ranking for error analysis
    # --------------------------------------------------------

    for _, result in (
        ranked.iterrows()
    ):

        ranking_rows.append({

            "question_id":
                question_id,

            "question_type":
                q[
                    "question_type"
                ],

            "route":
                q[
                    "route"
                ],

            "question":
                query,

            "trial_rank":
                int(
                    result[
                        "trial_rank"
                    ]
                ),

            "nct_id":
                result[
                    "nct_id"
                ],

            "is_gold_evidence":
                (
                    result[
                        "nct_id"
                    ]
                    in set(
                        gold_ids
                    )
                ),

            "bm25_score":
                float(
                    result[
                        "bm25_score"
                    ]
                ),

            "best_chunk_id":
                result[
                    "chunk_id"
                ],

            "best_chunk_type":
                result[
                    "chunk_type"
                ],

            "canonical_company":
                result[
                    "canonical_company"
                ],

            "brief_title":
                result[
                    "brief_title"
                ],

            "best_chunk_content":
                result[
                    "content"
                ],
        })


question_results = pd.DataFrame(
    question_rows
)

rankings = pd.DataFrame(
    ranking_rows
)


# ============================================================
# 8. Overall metrics
# ============================================================

overall = pd.DataFrame(
    [
        {
            "segment":
                "overall",

            "questions":
                len(
                    question_results
                ),

            "mean_recall_at_5":
                question_results[
                    "recall_at_5"
                ].mean(),

            "mean_recall_at_10":
                question_results[
                    "recall_at_10"
                ].mean(),

            "hit_rate_at_5":
                question_results[
                    "hit_at_5"
                ].mean(),

            "hit_rate_at_10":
                question_results[
                    "hit_at_10"
                ].mean(),

            "mrr":
                question_results[
                    "mrr"
                ].mean(),
        }
    ]
)


# ============================================================
# 9. Metrics by question type
# ============================================================

by_type = (

    question_results

    .groupby(
        "question_type"
    )

    .agg(
        questions=(
            "question_id",
            "count"
        ),

        mean_recall_at_5=(
            "recall_at_5",
            "mean"
        ),

        mean_recall_at_10=(
            "recall_at_10",
            "mean"
        ),

        hit_rate_at_5=(
            "hit_at_5",
            "mean"
        ),

        hit_rate_at_10=(
            "hit_at_10",
            "mean"
        ),

        mrr=(
            "mrr",
            "mean"
        ),
    )

    .reset_index()

    .rename(
        columns={
            "question_type":
                "segment"
        }
    )
)


# ============================================================
# 10. Metrics by route
# ============================================================

by_route = (

    question_results

    .groupby(
        "route"
    )

    .agg(
        questions=(
            "question_id",
            "count"
        ),

        mean_recall_at_5=(
            "recall_at_5",
            "mean"
        ),

        mean_recall_at_10=(
            "recall_at_10",
            "mean"
        ),

        hit_rate_at_5=(
            "hit_at_5",
            "mean"
        ),

        hit_rate_at_10=(
            "hit_at_10",
            "mean"
        ),

        mrr=(
            "mrr",
            "mean"
        ),
    )

    .reset_index()

    .assign(
        segment=lambda x:
            "route:"
            + x[
                "route"
            ]
    )

    .drop(
        columns=[
            "route"
        ]
    )
)


summary = pd.concat(
    [
        overall,
        by_type,
        by_route,
    ],
    ignore_index=True
)


# ============================================================
# 11. Validation
# ============================================================

assert len(
    question_results
) == 19


assert (
    question_results[
        "question_id"
    ].nunique()
    == 19
)


assert (
    rankings.groupby(
        "question_id"
    )[
        "nct_id"
    ]
    .apply(
        lambda x:
            x.is_unique
    )
    .all()
), (
    "Duplicate NCTs exist within ranked results."
)


assert (
    question_results[
        "recall_at_10"
    ]
    >=
    question_results[
        "recall_at_5"
    ]
).all()


# ============================================================
# 12. Save results
# ============================================================

question_results.to_csv(
    QUESTION_RESULTS_PATH,
    index=False
)


rankings.to_parquet(
    RANKINGS_PATH,
    index=False
)


summary.to_csv(
    SUMMARY_PATH,
    index=False
)


# ============================================================
# 13. Output
# ============================================================

print("\n")
print("=" * 100)
print("STAGE 4B — BM25 BASELINE")
print("=" * 100)


print(
    f"\nEvaluation questions: "
    f"{len(question_results)}"
)


print("\n")
print("=" * 100)
print("OVERALL METRICS")
print("=" * 100)


print(
    overall
    .round(3)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 100)
print("METRICS BY QUESTION TYPE")
print("=" * 100)


print(
    by_type
    .round(3)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 100)
print("METRICS BY ROUTE")
print("=" * 100)


print(
    by_route
    .round(3)
    .to_string(
        index=False
    )
)


# ============================================================
# 14. Hardest questions
# ============================================================

print("\n")
print("=" * 100)
print("LOWEST BM25 PERFORMANCE")
print("=" * 100)


hardest = (

    question_results[
        [
            "question_id",
            "question_type",
            "route",
            "gold_count",
            "recall_at_5",
            "recall_at_10",
            "mrr",
            "first_relevant_rank",
            "question",
        ]
    ]

    .sort_values(
        [
            "recall_at_10",
            "mrr",
        ],
        ascending=[
            True,
            True,
        ]
    )
)


print(
    hardest
    .head(10)
    .round(3)
    .to_string(
        index=False
    )
)


# ============================================================
# 15. First relevant rank distribution
# ============================================================

print("\n")
print("=" * 100)
print("FIRST RELEVANT RANK")
print("=" * 100)


print(
    question_results[
        "first_relevant_rank"
    ]
    .describe()
    .round(2)
    .to_string()
)


# ============================================================
# 16. Top-ranked chunk-type behaviour
# ============================================================

top1_chunk_types = (

    rankings.loc[
        rankings[
            "trial_rank"
        ]
        == 1,
        "best_chunk_type"
    ]

    .value_counts()
)


print("\n")
print("=" * 100)
print("TOP-1 BEST CHUNK TYPES")
print("=" * 100)


print(
    top1_chunk_types
    .to_string()
)


print("\n")
print("=" * 100)
print("STAGE 4B COMPLETE")
print("=" * 100)


print(
    f"\nQuestion metrics:\n"
    f"{QUESTION_RESULTS_PATH}"
)

print(
    f"\nTop-20 trial rankings:\n"
    f"{RANKINGS_PATH}"
)

print(
    f"\nSummary:\n"
    f"{SUMMARY_PATH}"
)

Building BM25 index...
Indexed chunks: 1,164


STAGE 4B — BM25 BASELINE

Evaluation questions: 19


OVERALL METRICS
segment  questions  mean_recall_at_5  mean_recall_at_10  hit_rate_at_5  hit_rate_at_10  mrr
overall         19              0.27               0.42          0.842             1.0 0.52


METRICS BY QUESTION TYPE
    segment  questions  mean_recall_at_5  mean_recall_at_10  hit_rate_at_5  hit_rate_at_10   mrr
 analytical         10             0.227              0.264            0.9             1.0 0.525
comparative          5             0.250              0.458            0.6             1.0 0.558
    factual          4             0.405              0.762            1.0             1.0 0.458


METRICS BY ROUTE
 questions  mean_recall_at_5  mean_recall_at_10  hit_rate_at_5  hit_rate_at_10   mrr         segment
         9             0.168              0.292          0.667             1.0 0.421    route:hybrid
        10             0.363              0.535          1.000  

In [5]:
# ============================================================
# STAGE 4C — DENSE RETRIEVAL BASELINE
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


# ============================================================
# 1. Paths
# ============================================================

PROJECT_ROOT = Path("..").resolve()

RETRIEVAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "retrieval"
)

EVAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "retrieval"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CHUNKS_PATH = (
    RETRIEVAL_DIR
    / "retrieval_chunks.parquet"
)

EVAL_PATH = (
    EVAL_DIR
    / "evaluation_questions_v2.csv"
)

BM25_RESULTS_PATH = (
    RESULTS_DIR
    / "bm25_question_metrics.csv"
)


DENSE_RESULTS_PATH = (
    RESULTS_DIR
    / "dense_question_metrics.csv"
)

DENSE_RANKINGS_PATH = (
    RESULTS_DIR
    / "dense_trial_rankings.parquet"
)

DENSE_SUMMARY_PATH = (
    RESULTS_DIR
    / "dense_summary.csv"
)

DENSE_COMPARISON_PATH = (
    RESULTS_DIR
    / "dense_vs_bm25.csv"
)

EMBEDDINGS_PATH = (
    RETRIEVAL_DIR
    / "bge_base_en_v1_5_chunk_embeddings.npy"
)


# ============================================================
# 2. Fixed dense retrieval configuration
#
# Important:
# Do NOT change the model after observing evaluation results.
# ============================================================

MODEL_NAME = "BAAI/bge-base-en-v1.5"

QUERY_PREFIX = (
    "Represent this sentence for searching relevant passages: "
)

BATCH_SIZE = 32


# ============================================================
# 3. Load frozen corpus + evaluation
# ============================================================

chunks = pd.read_parquet(
    CHUNKS_PATH
)

evaluation = pd.read_csv(
    EVAL_PATH
)

bm25_results = pd.read_csv(
    BM25_RESULTS_PATH
)


eval_rag = (
    evaluation.loc[
        evaluation[
            "requires_evidence_retrieval"
        ]
        == True
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(eval_rag) == 19


# ============================================================
# 4. Helpers
# ============================================================

def parse_json_list(x):

    if isinstance(x, list):
        return x

    if isinstance(x, str):

        try:
            parsed = json.loads(x)

            if isinstance(parsed, list):
                return parsed

        except Exception:
            pass

    return []


def recall_at_k(
    ranked_ids,
    gold_ids,
    k
):

    gold = set(
        gold_ids
    )

    if not gold:
        return np.nan

    retrieved = set(
        ranked_ids[:k]
    )

    return (
        len(
            gold & retrieved
        )
        /
        len(gold)
    )


def hit_at_k(
    ranked_ids,
    gold_ids,
    k
):

    gold = set(
        gold_ids
    )

    if not gold:
        return np.nan

    return float(
        bool(
            gold
            &
            set(
                ranked_ids[:k]
            )
        )
    )


def reciprocal_rank(
    ranked_ids,
    gold_ids
):

    gold = set(
        gold_ids
    )

    if not gold:
        return np.nan


    for rank, nct_id in enumerate(
        ranked_ids,
        start=1
    ):

        if nct_id in gold:

            return (
                1.0 / rank
            )

    return 0.0


# ============================================================
# 5. Load embedding model
# ============================================================

print(
    f"Loading dense retrieval model:\n"
    f"{MODEL_NAME}"
)


model = SentenceTransformer(
    MODEL_NAME
)


# ============================================================
# 6. Encode retrieval corpus
#
# Embeddings are normalized so cosine similarity
# becomes a simple dot product.
# ============================================================

if EMBEDDINGS_PATH.exists():

    print(
        "\nLoading cached chunk embeddings..."
    )

    chunk_embeddings = np.load(
        EMBEDDINGS_PATH
    )


    assert (
        chunk_embeddings.shape[0]
        ==
        len(chunks)
    ), (
        "Cached embedding count does not match "
        "current retrieval corpus."
    )


else:

    print(
        "\nEncoding retrieval corpus..."
    )

    chunk_embeddings = (
        model.encode(
            chunks[
                "content"
            ]
            .fillna("")
            .tolist(),

            batch_size=BATCH_SIZE,

            show_progress_bar=True,

            normalize_embeddings=True,

            convert_to_numpy=True,
        )
    )


    np.save(
        EMBEDDINGS_PATH,
        chunk_embeddings
    )


print(
    f"\nChunk embeddings shape: "
    f"{chunk_embeddings.shape}"
)


# ============================================================
# 7. Dense retrieval
#
# Search chunks first.
# Then collapse to unique NCT using highest chunk similarity.
# ============================================================

def rank_trials_dense(
    query,
    top_trials=20
):

    query_text = (
        QUERY_PREFIX
        +
        str(query)
    )


    query_embedding = (
        model.encode(
            [query_text],

            normalize_embeddings=True,

            convert_to_numpy=True,
        )[0]
    )


    # cosine similarity because embeddings normalized
    scores = (
        chunk_embeddings
        @ query_embedding
    )


    ranked_chunks = (
        chunks[
            [
                "chunk_id",
                "nct_id",
                "chunk_type",
                "canonical_company",
                "brief_title",
                "content",
            ]
        ]
        .copy()
    )


    ranked_chunks[
        "dense_score"
    ] = scores


    ranked_chunks = (
        ranked_chunks
        .sort_values(
            [
                "dense_score",
                "chunk_id",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .reset_index(drop=True)
    )


    # Best-scoring chunk represents trial
    ranked_trials = (
        ranked_chunks
        .drop_duplicates(
            subset="nct_id",
            keep="first"
        )
        .reset_index(drop=True)
    )


    ranked_trials[
        "trial_rank"
    ] = (
        np.arange(
            len(ranked_trials)
        )
        + 1
    )


    return (
        ranked_trials
        .head(top_trials)
        .copy()
    )


# ============================================================
# 8. Evaluate all 19 retrieval/hybrid questions
# ============================================================

question_rows = []
ranking_rows = []


for _, q in eval_rag.iterrows():

    qid = q[
        "question_id"
    ]

    query = q[
        "question"
    ]

    gold_ids = (
        parse_json_list(
            q[
                "gold_evidence_nct_ids"
            ]
        )
    )

    gold_set = set(
        gold_ids
    )


    assert len(gold_ids) > 0


    ranked = rank_trials_dense(
        query=query,
        top_trials=20
    )


    ranked_ids = (
        ranked[
            "nct_id"
        ]
        .tolist()
    )


    relevant_ranks = [

        rank

        for rank, nct_id
        in enumerate(
            ranked_ids,
            start=1
        )

        if nct_id in gold_set
    ]


    first_relevant_rank = (

        min(
            relevant_ranks
        )

        if relevant_ranks

        else None
    )


    question_rows.append({

        "question_id":
            qid,

        "question_type":
            q[
                "question_type"
            ],

        "route":
            q[
                "route"
            ],

        "question":
            query,

        "gold_count":
            len(
                gold_ids
            ),

        "recall_at_5":
            recall_at_k(
                ranked_ids,
                gold_ids,
                5
            ),

        "recall_at_10":
            recall_at_k(
                ranked_ids,
                gold_ids,
                10
            ),

        "hit_at_5":
            hit_at_k(
                ranked_ids,
                gold_ids,
                5
            ),

        "hit_at_10":
            hit_at_k(
                ranked_ids,
                gold_ids,
                10
            ),

        "mrr":
            reciprocal_rank(
                ranked_ids,
                gold_ids
            ),

        "first_relevant_rank":
            first_relevant_rank,

        "top_1_nct":
            (
                ranked_ids[0]
                if ranked_ids
                else None
            ),

        "top_1_is_gold":
            bool(
                ranked_ids
                and
                ranked_ids[0]
                in gold_set
            ),
    })


    # ========================================================
    # Save top-20 rankings for error analysis
    # ========================================================

    for _, result in (
        ranked.iterrows()
    ):

        ranking_rows.append({

            "question_id":
                qid,

            "question_type":
                q[
                    "question_type"
                ],

            "route":
                q[
                    "route"
                ],

            "question":
                query,

            "trial_rank":
                int(
                    result[
                        "trial_rank"
                    ]
                ),

            "nct_id":
                result[
                    "nct_id"
                ],

            "is_gold_evidence":
                (
                    result[
                        "nct_id"
                    ]
                    in gold_set
                ),

            "dense_score":
                float(
                    result[
                        "dense_score"
                    ]
                ),

            "best_chunk_id":
                result[
                    "chunk_id"
                ],

            "best_chunk_type":
                result[
                    "chunk_type"
                ],

            "canonical_company":
                result[
                    "canonical_company"
                ],

            "brief_title":
                result[
                    "brief_title"
                ],

            "best_chunk_content":
                result[
                    "content"
                ],
        })


dense_results = pd.DataFrame(
    question_rows
)

dense_rankings = pd.DataFrame(
    ranking_rows
)


# ============================================================
# 9. Overall summary
# ============================================================

overall = pd.DataFrame(
    [
        {
            "segment":
                "overall",

            "questions":
                len(
                    dense_results
                ),

            "mean_recall_at_5":
                dense_results[
                    "recall_at_5"
                ].mean(),

            "mean_recall_at_10":
                dense_results[
                    "recall_at_10"
                ].mean(),

            "hit_rate_at_5":
                dense_results[
                    "hit_at_5"
                ].mean(),

            "hit_rate_at_10":
                dense_results[
                    "hit_at_10"
                ].mean(),

            "mrr":
                dense_results[
                    "mrr"
                ].mean(),
        }
    ]
)


# ============================================================
# 10. By question type
# ============================================================

by_type = (

    dense_results

    .groupby(
        "question_type"
    )

    .agg(
        questions=(
            "question_id",
            "count"
        ),

        mean_recall_at_5=(
            "recall_at_5",
            "mean"
        ),

        mean_recall_at_10=(
            "recall_at_10",
            "mean"
        ),

        hit_rate_at_5=(
            "hit_at_5",
            "mean"
        ),

        hit_rate_at_10=(
            "hit_at_10",
            "mean"
        ),

        mrr=(
            "mrr",
            "mean"
        ),
    )

    .reset_index()

    .rename(
        columns={
            "question_type":
                "segment"
        }
    )
)


# ============================================================
# 11. By route
# ============================================================

by_route = (

    dense_results

    .groupby(
        "route"
    )

    .agg(
        questions=(
            "question_id",
            "count"
        ),

        mean_recall_at_5=(
            "recall_at_5",
            "mean"
        ),

        mean_recall_at_10=(
            "recall_at_10",
            "mean"
        ),

        hit_rate_at_5=(
            "hit_at_5",
            "mean"
        ),

        hit_rate_at_10=(
            "hit_at_10",
            "mean"
        ),

        mrr=(
            "mrr",
            "mean"
        ),
    )

    .reset_index()
)


by_route[
    "segment"
] = (
    "route:"
    +
    by_route[
        "route"
    ]
)


by_route = (
    by_route.drop(
        columns="route"
    )
)


summary = pd.concat(
    [
        overall,
        by_type,
        by_route,
    ],
    ignore_index=True
)


# ============================================================
# 12. BM25 comparison
# ============================================================

comparison = (

    dense_results[
        [
            "question_id",
            "question_type",
            "route",
            "question",
            "recall_at_5",
            "recall_at_10",
            "mrr",
            "first_relevant_rank",
        ]
    ]

    .merge(

        bm25_results[
            [
                "question_id",
                "recall_at_5",
                "recall_at_10",
                "mrr",
                "first_relevant_rank",
            ]
        ],

        on="question_id",

        how="left",

        suffixes=(
            "_dense",
            "_bm25"
        ),

        validate="one_to_one"
    )
)


comparison[
    "delta_recall_at_5"
] = (
    comparison[
        "recall_at_5_dense"
    ]
    -
    comparison[
        "recall_at_5_bm25"
    ]
)


comparison[
    "delta_recall_at_10"
] = (
    comparison[
        "recall_at_10_dense"
    ]
    -
    comparison[
        "recall_at_10_bm25"
    ]
)


comparison[
    "delta_mrr"
] = (
    comparison[
        "mrr_dense"
    ]
    -
    comparison[
        "mrr_bm25"
    ]
)


# ============================================================
# 13. Validation
# ============================================================

assert len(
    dense_results
) == 19


assert (
    dense_results[
        "question_id"
    ].nunique()
    == 19
)


assert (
    dense_results[
        "recall_at_10"
    ]
    >=
    dense_results[
        "recall_at_5"
    ]
).all()


assert (

    dense_rankings

    .groupby(
        "question_id"
    )[
        "nct_id"
    ]

    .apply(
        lambda x:
            x.is_unique
    )

    .all()

), (
    "Duplicate trials found after NCT collapse."
)


# ============================================================
# 14. Save
# ============================================================

dense_results.to_csv(
    DENSE_RESULTS_PATH,
    index=False
)

dense_rankings.to_parquet(
    DENSE_RANKINGS_PATH,
    index=False
)

summary.to_csv(
    DENSE_SUMMARY_PATH,
    index=False
)

comparison.to_csv(
    DENSE_COMPARISON_PATH,
    index=False
)


# ============================================================
# 15. Output
# ============================================================

print("\n")
print("=" * 100)
print("STAGE 4C — DENSE RETRIEVAL")
print("=" * 100)


print(
    f"\nModel: "
    f"{MODEL_NAME}"
)

print(
    f"Evaluation questions: "
    f"{len(dense_results)}"
)


# ============================================================
# Overall
# ============================================================

print("\n")
print("=" * 100)
print("OVERALL METRICS")
print("=" * 100)

print(
    overall
    .round(3)
    .to_string(
        index=False
    )
)


# ============================================================
# Type
# ============================================================

print("\n")
print("=" * 100)
print("METRICS BY QUESTION TYPE")
print("=" * 100)

print(
    by_type
    .round(3)
    .to_string(
        index=False
    )
)


# ============================================================
# Route
# ============================================================

print("\n")
print("=" * 100)
print("METRICS BY ROUTE")
print("=" * 100)

print(
    by_route
    .round(3)
    .to_string(
        index=False
    )
)


# ============================================================
# BM25 vs Dense aggregate
# ============================================================

print("\n")
print("=" * 100)
print("BM25 vs DENSE — OVERALL")
print("=" * 100)


aggregate_comparison = pd.DataFrame({

    "metric": [
        "Recall@5",
        "Recall@10",
        "MRR",
    ],

    "BM25": [

        bm25_results[
            "recall_at_5"
        ].mean(),

        bm25_results[
            "recall_at_10"
        ].mean(),

        bm25_results[
            "mrr"
        ].mean(),
    ],

    "Dense": [

        dense_results[
            "recall_at_5"
        ].mean(),

        dense_results[
            "recall_at_10"
        ].mean(),

        dense_results[
            "mrr"
        ].mean(),
    ],
})


aggregate_comparison[
    "delta"
] = (
    aggregate_comparison[
        "Dense"
    ]
    -
    aggregate_comparison[
        "BM25"
    ]
)


print(
    aggregate_comparison
    .round(3)
    .to_string(
        index=False
    )
)


# ============================================================
# Biggest dense improvements
# ============================================================

print("\n")
print("=" * 100)
print("BIGGEST DENSE IMPROVEMENTS")
print("=" * 100)


print(

    comparison[
        [
            "question_id",
            "question_type",
            "route",
            "recall_at_10_bm25",
            "recall_at_10_dense",
            "delta_recall_at_10",
            "mrr_bm25",
            "mrr_dense",
            "delta_mrr",
            "question",
        ]
    ]

    .sort_values(
        [
            "delta_recall_at_10",
            "delta_mrr",
        ],
        ascending=False
    )

    .head(10)

    .round(3)

    .to_string(
        index=False
    )
)


# ============================================================
# Biggest regressions
# ============================================================

print("\n")
print("=" * 100)
print("BIGGEST DENSE REGRESSIONS")
print("=" * 100)


print(

    comparison[
        [
            "question_id",
            "question_type",
            "route",
            "recall_at_10_bm25",
            "recall_at_10_dense",
            "delta_recall_at_10",
            "mrr_bm25",
            "mrr_dense",
            "delta_mrr",
            "question",
        ]
    ]

    .sort_values(
        [
            "delta_recall_at_10",
            "delta_mrr",
        ],
        ascending=True
    )

    .head(10)

    .round(3)

    .to_string(
        index=False
    )
)


# ============================================================
# Hardest dense questions
# ============================================================

print("\n")
print("=" * 100)
print("LOWEST DENSE PERFORMANCE")
print("=" * 100)


print(

    dense_results[
        [
            "question_id",
            "question_type",
            "route",
            "gold_count",
            "recall_at_5",
            "recall_at_10",
            "mrr",
            "first_relevant_rank",
            "question",
        ]
    ]

    .sort_values(
        [
            "recall_at_10",
            "mrr",
        ],
        ascending=[
            True,
            True,
        ]
    )

    .head(10)

    .round(3)

    .to_string(
        index=False
    )
)


# ============================================================
# First relevant rank
# ============================================================

print("\n")
print("=" * 100)
print("FIRST RELEVANT RANK")
print("=" * 100)


print(
    dense_results[
        "first_relevant_rank"
    ]
    .describe()
    .round(2)
    .to_string()
)


# ============================================================
# Top chunk types
# ============================================================

print("\n")
print("=" * 100)
print("TOP-1 BEST CHUNK TYPES")
print("=" * 100)


print(

    dense_rankings.loc[
        dense_rankings[
            "trial_rank"
        ]
        == 1,
        "best_chunk_type"
    ]

    .value_counts()

    .to_string()
)


# ============================================================
# Complete
# ============================================================

print("\n")
print("=" * 100)
print("STAGE 4C COMPLETE")
print("=" * 100)


print(
    f"\nDense question metrics:\n"
    f"{DENSE_RESULTS_PATH}"
)

print(
    f"\nDense trial rankings:\n"
    f"{DENSE_RANKINGS_PATH}"
)

print(
    f"\nDense vs BM25 comparison:\n"
    f"{DENSE_COMPARISON_PATH}"
)

c:\Users\shubh\Desktop\Projects\Copilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dense retrieval model:
BAAI/bge-base-en-v1.5


c:\Users\shubh\Desktop\Projects\Copilot\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shubh\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2164.25it/s]



Encoding retrieval corpus...


Batches: 100%|██████████| 37/37 [05:40<00:00,  9.20s/it]



Chunk embeddings shape: (1164, 768)


STAGE 4C — DENSE RETRIEVAL

Model: BAAI/bge-base-en-v1.5
Evaluation questions: 19


OVERALL METRICS
segment  questions  mean_recall_at_5  mean_recall_at_10  hit_rate_at_5  hit_rate_at_10   mrr
overall         19             0.241              0.378          0.579           0.632 0.424


METRICS BY QUESTION TYPE
    segment  questions  mean_recall_at_5  mean_recall_at_10  hit_rate_at_5  hit_rate_at_10   mrr
 analytical         10             0.060              0.093            0.3             0.4 0.214
comparative          5             0.342              0.508            0.8             0.8 0.650
    factual          4             0.565              0.929            1.0             1.0 0.667


METRICS BY ROUTE
 questions  mean_recall_at_5  mean_recall_at_10  hit_rate_at_5  hit_rate_at_10   mrr         segment
         9              0.13              0.227          0.444           0.556 0.322    route:hybrid
        10              0.34           

In [7]:
# ============================================================
# STAGE 4D — HYBRID RETRIEVAL WITH RECIPROCAL RANK FUSION
# FULL REPLACEMENT
# ============================================================

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


# ============================================================
# 1. Paths
# ============================================================

PROJECT_ROOT = Path("..").resolve()

RETRIEVAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "retrieval"
)

EVAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "retrieval"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CHUNKS_PATH = (
    RETRIEVAL_DIR
    / "retrieval_chunks.parquet"
)

EVAL_PATH = (
    EVAL_DIR
    / "evaluation_questions_v2.csv"
)

BM25_RESULTS_PATH = (
    RESULTS_DIR
    / "bm25_question_metrics.csv"
)

DENSE_RESULTS_PATH = (
    RESULTS_DIR
    / "dense_question_metrics.csv"
)

EMBEDDINGS_PATH = (
    RETRIEVAL_DIR
    / "bge_base_en_v1_5_chunk_embeddings.npy"
)


HYBRID_RESULTS_PATH = (
    RESULTS_DIR
    / "hybrid_rrf_question_metrics.csv"
)

HYBRID_RANKINGS_PATH = (
    RESULTS_DIR
    / "hybrid_rrf_trial_rankings.parquet"
)

HYBRID_SUMMARY_PATH = (
    RESULTS_DIR
    / "hybrid_rrf_summary.csv"
)

HYBRID_COMPARISON_PATH = (
    RESULTS_DIR
    / "hybrid_rrf_vs_baselines.csv"
)


# ============================================================
# 2. Frozen configuration
# ============================================================

MODEL_NAME = "BAAI/bge-base-en-v1.5"

QUERY_PREFIX = (
    "Represent this sentence for searching relevant passages: "
)

# Standard fixed RRF constant.
# Do not tune against this benchmark.
RRF_K = 60


# ============================================================
# 3. Load frozen assets
# ============================================================

chunks = pd.read_parquet(
    CHUNKS_PATH
)

evaluation = pd.read_csv(
    EVAL_PATH
)

bm25_results = pd.read_csv(
    BM25_RESULTS_PATH
)

dense_results = pd.read_csv(
    DENSE_RESULTS_PATH
)


eval_rag = (
    evaluation.loc[
        evaluation[
            "requires_evidence_retrieval"
        ]
        == True
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(eval_rag) == 19


# ============================================================
# 4. Helpers
# ============================================================

def parse_json_list(x):

    if isinstance(x, list):
        return x

    if isinstance(x, str):

        try:

            parsed = json.loads(x)

            if isinstance(parsed, list):
                return parsed

        except Exception:
            pass

    return []


TOKEN_PATTERN = re.compile(
    r"[a-z0-9]+(?:-[a-z0-9]+)*"
)


def tokenize(text):

    return TOKEN_PATTERN.findall(
        str(text).lower()
    )


def recall_at_k(
    ranked_ids,
    gold_ids,
    k
):

    gold = set(
        gold_ids
    )

    if not gold:
        return np.nan

    retrieved = set(
        ranked_ids[:k]
    )

    return (
        len(
            gold & retrieved
        )
        /
        len(gold)
    )


def hit_at_k(
    ranked_ids,
    gold_ids,
    k
):

    gold = set(
        gold_ids
    )

    if not gold:
        return np.nan

    return float(
        bool(
            gold
            &
            set(
                ranked_ids[:k]
            )
        )
    )


def reciprocal_rank(
    ranked_ids,
    gold_ids
):

    gold = set(
        gold_ids
    )

    if not gold:
        return np.nan

    for rank, nct_id in enumerate(
        ranked_ids,
        start=1
    ):

        if nct_id in gold:
            return 1.0 / rank

    return 0.0


# ============================================================
# 5. Rebuild BM25 baseline
# ============================================================

tokenized_corpus = (
    chunks[
        "content"
    ]
    .fillna("")
    .apply(tokenize)
    .tolist()
)


bm25 = BM25Okapi(
    tokenized_corpus
)


# ============================================================
# 6. Load dense model + frozen embeddings
# ============================================================

model = SentenceTransformer(
    MODEL_NAME
)


chunk_embeddings = np.load(
    EMBEDDINGS_PATH
)


assert (
    chunk_embeddings.shape[0]
    ==
    len(chunks)
), (
    "Dense embedding count does not match "
    "current chunk corpus."
)


# ============================================================
# 7. Full trial rankings
#
# Retrieve at chunk level.
# Collapse to NCT using best-scoring chunk.
# ============================================================

TRIAL_COLUMNS = [
    "chunk_id",
    "nct_id",
    "chunk_type",
    "canonical_company",
    "brief_title",
    "content",
]


def full_bm25_trial_ranking(query):

    scores = bm25.get_scores(
        tokenize(query)
    )


    ranked_chunks = (
        chunks[
            TRIAL_COLUMNS
        ]
        .copy()
    )


    ranked_chunks[
        "bm25_score"
    ] = scores


    ranked_chunks = (
        ranked_chunks
        .sort_values(
            [
                "bm25_score",
                "chunk_id",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .reset_index(drop=True)
    )


    ranked_trials = (
        ranked_chunks
        .drop_duplicates(
            subset="nct_id",
            keep="first"
        )
        .reset_index(drop=True)
    )


    ranked_trials[
        "bm25_rank"
    ] = (
        np.arange(
            len(ranked_trials)
        )
        + 1
    )


    return ranked_trials


def full_dense_trial_ranking(query):

    query_embedding = (
        model.encode(
            [
                QUERY_PREFIX
                +
                str(query)
            ],
            normalize_embeddings=True,
            convert_to_numpy=True,
        )[0]
    )


    scores = (
        chunk_embeddings
        @ query_embedding
    )


    ranked_chunks = (
        chunks[
            TRIAL_COLUMNS
        ]
        .copy()
    )


    ranked_chunks[
        "dense_score"
    ] = scores


    ranked_chunks = (
        ranked_chunks
        .sort_values(
            [
                "dense_score",
                "chunk_id",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .reset_index(drop=True)
    )


    ranked_trials = (
        ranked_chunks
        .drop_duplicates(
            subset="nct_id",
            keep="first"
        )
        .reset_index(drop=True)
    )


    ranked_trials[
        "dense_rank"
    ] = (
        np.arange(
            len(ranked_trials)
        )
        + 1
    )


    return ranked_trials


# ============================================================
# 8. Reciprocal Rank Fusion
# ============================================================

def rank_trials_rrf(
    query,
    top_trials=20
):

    bm25_ranked = (
        full_bm25_trial_ranking(
            query
        )
    )

    dense_ranked = (
        full_dense_trial_ranking(
            query
        )
    )


    bm25_side = (
        bm25_ranked[
            [
                "nct_id",
                "bm25_rank",
                "bm25_score",
                "chunk_id",
                "chunk_type",
                "content",
                "canonical_company",
                "brief_title",
            ]
        ]
        .rename(
            columns={
                "chunk_id":
                    "bm25_best_chunk_id",

                "chunk_type":
                    "bm25_best_chunk_type",

                "content":
                    "bm25_best_chunk_content",
            }
        )
    )


    dense_side = (
        dense_ranked[
            [
                "nct_id",
                "dense_rank",
                "dense_score",
                "chunk_id",
                "chunk_type",
                "content",
            ]
        ]
        .rename(
            columns={
                "chunk_id":
                    "dense_best_chunk_id",

                "chunk_type":
                    "dense_best_chunk_type",

                "content":
                    "dense_best_chunk_content",
            }
        )
    )


    fused = (
        bm25_side
        .merge(
            dense_side,
            on="nct_id",
            how="inner",
            validate="one_to_one",
        )
    )


    assert (
        len(fused)
        ==
        chunks[
            "nct_id"
        ].nunique()
    )


    fused[
        "rrf_bm25_component"
    ] = (
        1.0
        /
        (
            RRF_K
            +
            fused[
                "bm25_rank"
            ]
        )
    )


    fused[
        "rrf_dense_component"
    ] = (
        1.0
        /
        (
            RRF_K
            +
            fused[
                "dense_rank"
            ]
        )
    )


    fused[
        "rrf_score"
    ] = (
        fused[
            "rrf_bm25_component"
        ]
        +
        fused[
            "rrf_dense_component"
        ]
    )


    fused = (
        fused
        .sort_values(
            [
                "rrf_score",
                "bm25_rank",
                "dense_rank",
                "nct_id",
            ],
            ascending=[
                False,
                True,
                True,
                True,
            ]
        )
        .reset_index(drop=True)
    )


    fused[
        "trial_rank"
    ] = (
        np.arange(
            len(fused)
        )
        + 1
    )


    return (
        fused
        .head(top_trials)
        .copy()
    )


# ============================================================
# 9. Run hybrid evaluation
# ============================================================

question_rows = []
ranking_rows = []


for _, q in eval_rag.iterrows():

    qid = q[
        "question_id"
    ]

    query = q[
        "question"
    ]

    gold_ids = parse_json_list(
        q[
            "gold_evidence_nct_ids"
        ]
    )

    gold_set = set(
        gold_ids
    )


    assert len(gold_ids) > 0


    ranked = rank_trials_rrf(
        query=query,
        top_trials=20
    )


    ranked_ids = (
        ranked[
            "nct_id"
        ]
        .tolist()
    )


    relevant_ranks = [

        rank

        for rank, nct_id
        in enumerate(
            ranked_ids,
            start=1
        )

        if nct_id in gold_set
    ]


    first_relevant_rank = (

        min(
            relevant_ranks
        )

        if relevant_ranks

        else None
    )


    question_rows.append({

        "question_id":
            qid,

        "question_type":
            q[
                "question_type"
            ],

        "route":
            q[
                "route"
            ],

        "question":
            query,

        "gold_count":
            len(
                gold_ids
            ),

        "recall_at_5":
            recall_at_k(
                ranked_ids,
                gold_ids,
                5
            ),

        "recall_at_10":
            recall_at_k(
                ranked_ids,
                gold_ids,
                10
            ),

        "hit_at_5":
            hit_at_k(
                ranked_ids,
                gold_ids,
                5
            ),

        "hit_at_10":
            hit_at_k(
                ranked_ids,
                gold_ids,
                10
            ),

        "mrr":
            reciprocal_rank(
                ranked_ids,
                gold_ids
            ),

        "first_relevant_rank":
            first_relevant_rank,

        "top_1_nct":
            (
                ranked_ids[0]
                if ranked_ids
                else None
            ),

        "top_1_is_gold":
            bool(
                ranked_ids
                and
                ranked_ids[0]
                in gold_set
            ),
    })


    for _, r in (
        ranked.iterrows()
    ):

        ranking_rows.append({

            "question_id":
                qid,

            "question_type":
                q[
                    "question_type"
                ],

            "route":
                q[
                    "route"
                ],

            "question":
                query,

            "trial_rank":
                int(
                    r[
                        "trial_rank"
                    ]
                ),

            "nct_id":
                r[
                    "nct_id"
                ],

            "is_gold_evidence":
                (
                    r[
                        "nct_id"
                    ]
                    in gold_set
                ),

            "rrf_score":
                float(
                    r[
                        "rrf_score"
                    ]
                ),

            "bm25_rank":
                int(
                    r[
                        "bm25_rank"
                    ]
                ),

            "dense_rank":
                int(
                    r[
                        "dense_rank"
                    ]
                ),

            "rrf_bm25_component":
                float(
                    r[
                        "rrf_bm25_component"
                    ]
                ),

            "rrf_dense_component":
                float(
                    r[
                        "rrf_dense_component"
                    ]
                ),

            "canonical_company":
                r[
                    "canonical_company"
                ],

            "brief_title":
                r[
                    "brief_title"
                ],

            "bm25_best_chunk_id":
                r[
                    "bm25_best_chunk_id"
                ],

            "bm25_best_chunk_type":
                r[
                    "bm25_best_chunk_type"
                ],

            "dense_best_chunk_id":
                r[
                    "dense_best_chunk_id"
                ],

            "dense_best_chunk_type":
                r[
                    "dense_best_chunk_type"
                ],
        })


hybrid_results = pd.DataFrame(
    question_rows
)

hybrid_rankings = pd.DataFrame(
    ranking_rows
)


# ============================================================
# 10. Summary helper
# ============================================================

def metric_summary(
    df,
    segment_name
):

    return {

        "segment":
            segment_name,

        "questions":
            len(df),

        "mean_recall_at_5":
            df[
                "recall_at_5"
            ].mean(),

        "mean_recall_at_10":
            df[
                "recall_at_10"
            ].mean(),

        "hit_rate_at_5":
            df[
                "hit_at_5"
            ].mean(),

        "hit_rate_at_10":
            df[
                "hit_at_10"
            ].mean(),

        "mrr":
            df[
                "mrr"
            ].mean(),
    }


summary_rows = [
    metric_summary(
        hybrid_results,
        "overall"
    )
]


for question_type, group in (
    hybrid_results.groupby(
        "question_type"
    )
):

    summary_rows.append(
        metric_summary(
            group,
            question_type
        )
    )


for route, group in (
    hybrid_results.groupby(
        "route"
    )
):

    summary_rows.append(
        metric_summary(
            group,
            f"route:{route}"
        )
    )


summary = pd.DataFrame(
    summary_rows
)


# ============================================================
# 11. Three-way comparison
# ============================================================

comparison = (

    hybrid_results[
        [
            "question_id",
            "question_type",
            "route",
            "question",
            "recall_at_5",
            "recall_at_10",
            "mrr",
        ]
    ]

    .rename(
        columns={
            "recall_at_5":
                "recall_at_5_rrf",

            "recall_at_10":
                "recall_at_10_rrf",

            "mrr":
                "mrr_rrf",
        }
    )

    .merge(

        bm25_results[
            [
                "question_id",
                "recall_at_5",
                "recall_at_10",
                "mrr",
            ]
        ]
        .rename(
            columns={
                "recall_at_5":
                    "recall_at_5_bm25",

                "recall_at_10":
                    "recall_at_10_bm25",

                "mrr":
                    "mrr_bm25",
            }
        ),

        on="question_id",

        how="left",

        validate="one_to_one",
    )

    .merge(

        dense_results[
            [
                "question_id",
                "recall_at_5",
                "recall_at_10",
                "mrr",
            ]
        ]
        .rename(
            columns={
                "recall_at_5":
                    "recall_at_5_dense",

                "recall_at_10":
                    "recall_at_10_dense",

                "mrr":
                    "mrr_dense",
            }
        ),

        on="question_id",

        how="left",

        validate="one_to_one",
    )
)


# ============================================================
# 12. Explicit deltas
# ============================================================

comparison[
    "rrf_delta_vs_bm25_recall5"
] = (
    comparison[
        "recall_at_5_rrf"
    ]
    -
    comparison[
        "recall_at_5_bm25"
    ]
)


comparison[
    "rrf_delta_vs_bm25_recall10"
] = (
    comparison[
        "recall_at_10_rrf"
    ]
    -
    comparison[
        "recall_at_10_bm25"
    ]
)


comparison[
    "rrf_delta_vs_dense_recall10"
] = (
    comparison[
        "recall_at_10_rrf"
    ]
    -
    comparison[
        "recall_at_10_dense"
    ]
)


comparison[
    "rrf_delta_vs_bm25_mrr"
] = (
    comparison[
        "mrr_rrf"
    ]
    -
    comparison[
        "mrr_bm25"
    ]
)


comparison[
    "rrf_delta_vs_dense_mrr"
] = (
    comparison[
        "mrr_rrf"
    ]
    -
    comparison[
        "mrr_dense"
    ]
)


# ============================================================
# 13. Validation
# ============================================================

assert len(
    hybrid_results
) == 19


assert (
    hybrid_results[
        "question_id"
    ].nunique()
    == 19
)


assert (
    hybrid_results[
        "recall_at_10"
    ]
    >=
    hybrid_results[
        "recall_at_5"
    ]
).all()


assert (

    hybrid_rankings

    .groupby(
        "question_id"
    )[
        "nct_id"
    ]

    .apply(
        lambda x:
            x.is_unique
    )

    .all()

), (
    "Duplicate NCTs exist after fusion."
)


# ============================================================
# 14. Save outputs
# ============================================================

hybrid_results.to_csv(
    HYBRID_RESULTS_PATH,
    index=False
)

hybrid_rankings.to_parquet(
    HYBRID_RANKINGS_PATH,
    index=False
)

summary.to_csv(
    HYBRID_SUMMARY_PATH,
    index=False
)

comparison.to_csv(
    HYBRID_COMPARISON_PATH,
    index=False
)


# ============================================================
# 15. Main output
# ============================================================

print("\n")
print("=" * 100)
print("STAGE 4D — HYBRID RRF")
print("=" * 100)


print(
    f"\nRRF k: {RRF_K}"
)

print(
    f"Evaluation questions: "
    f"{len(hybrid_results)}"
)


# ============================================================
# 16. Overall metrics
# ============================================================

print("\n")
print("=" * 100)
print("OVERALL METRICS")
print("=" * 100)


print(

    summary.loc[
        summary[
            "segment"
        ]
        == "overall"
    ]

    .round(3)

    .to_string(
        index=False
    )
)


# ============================================================
# 17. By question type
# ============================================================

print("\n")
print("=" * 100)
print("METRICS BY QUESTION TYPE")
print("=" * 100)


print(

    summary.loc[
        summary[
            "segment"
        ]
        .isin(
            [
                "factual",
                "comparative",
                "analytical",
            ]
        )
    ]

    .round(3)

    .to_string(
        index=False
    )
)


# ============================================================
# 18. By route
# ============================================================

print("\n")
print("=" * 100)
print("METRICS BY ROUTE")
print("=" * 100)


print(

    summary.loc[
        summary[
            "segment"
        ]
        .str.startswith(
            "route:"
        )
    ]

    .round(3)

    .to_string(
        index=False
    )
)


# ============================================================
# 19. Aggregate three-way comparison
# ============================================================

aggregate_comparison = pd.DataFrame({

    "metric": [
        "Recall@5",
        "Recall@10",
        "MRR",
    ],

    "BM25": [

        bm25_results[
            "recall_at_5"
        ].mean(),

        bm25_results[
            "recall_at_10"
        ].mean(),

        bm25_results[
            "mrr"
        ].mean(),
    ],

    "Dense": [

        dense_results[
            "recall_at_5"
        ].mean(),

        dense_results[
            "recall_at_10"
        ].mean(),

        dense_results[
            "mrr"
        ].mean(),
    ],

    "Hybrid_RRF": [

        hybrid_results[
            "recall_at_5"
        ].mean(),

        hybrid_results[
            "recall_at_10"
        ].mean(),

        hybrid_results[
            "mrr"
        ].mean(),
    ],
})


aggregate_comparison[
    "RRF_vs_BM25"
] = (
    aggregate_comparison[
        "Hybrid_RRF"
    ]
    -
    aggregate_comparison[
        "BM25"
    ]
)


aggregate_comparison[
    "RRF_vs_Dense"
] = (
    aggregate_comparison[
        "Hybrid_RRF"
    ]
    -
    aggregate_comparison[
        "Dense"
    ]
)


print("\n")
print("=" * 100)
print("BM25 vs DENSE vs HYBRID RRF")
print("=" * 100)


print(
    aggregate_comparison
    .round(3)
    .to_string(
        index=False
    )
)


# ============================================================
# 20. Biggest RRF improvements vs BM25
#
# FIX:
# sort on full comparison DataFrame BEFORE selecting display cols.
# ============================================================

print("\n")
print("=" * 100)
print("BIGGEST RRF IMPROVEMENTS OVER BM25")
print("=" * 100)


rrf_improvements = (

    comparison

    .sort_values(
        [
            "rrf_delta_vs_bm25_recall10",
            "rrf_delta_vs_bm25_mrr",
        ],
        ascending=[
            False,
            False,
        ]
    )

    .head(10)
)


print(

    rrf_improvements[
        [
            "question_id",
            "question_type",
            "route",
            "recall_at_10_bm25",
            "recall_at_10_dense",
            "recall_at_10_rrf",
            "rrf_delta_vs_bm25_recall10",
            "mrr_bm25",
            "mrr_rrf",
            "rrf_delta_vs_bm25_mrr",
            "question",
        ]
    ]

    .round(3)

    .to_string(
        index=False
    )
)


# ============================================================
# 21. Biggest RRF regressions vs BM25
# ============================================================

print("\n")
print("=" * 100)
print("BIGGEST RRF REGRESSIONS VS BM25")
print("=" * 100)


rrf_regressions = (

    comparison

    .sort_values(
        [
            "rrf_delta_vs_bm25_recall10",
            "rrf_delta_vs_bm25_mrr",
        ],
        ascending=[
            True,
            True,
        ]
    )

    .head(10)
)


print(

    rrf_regressions[
        [
            "question_id",
            "question_type",
            "route",
            "recall_at_10_bm25",
            "recall_at_10_dense",
            "recall_at_10_rrf",
            "rrf_delta_vs_bm25_recall10",
            "mrr_bm25",
            "mrr_rrf",
            "rrf_delta_vs_bm25_mrr",
            "question",
        ]
    ]

    .round(3)

    .to_string(
        index=False
    )
)


# ============================================================
# 22. Lowest hybrid performance
# ============================================================

print("\n")
print("=" * 100)
print("LOWEST HYBRID PERFORMANCE")
print("=" * 100)


lowest_hybrid = (

    hybrid_results

    .sort_values(
        [
            "recall_at_10",
            "mrr",
        ],
        ascending=[
            True,
            True,
        ]
    )

    .head(10)
)


print(

    lowest_hybrid[
        [
            "question_id",
            "question_type",
            "route",
            "gold_count",
            "recall_at_5",
            "recall_at_10",
            "mrr",
            "first_relevant_rank",
            "question",
        ]
    ]

    .round(3)

    .to_string(
        index=False
    )
)


# ============================================================
# 23. Win / tie / loss against BM25
# ============================================================

comparison[
    "rrf_vs_bm25_recall10_result"
] = np.select(

    [
        comparison[
            "rrf_delta_vs_bm25_recall10"
        ] > 0,

        comparison[
            "rrf_delta_vs_bm25_recall10"
        ] < 0,
    ],

    [
        "win",
        "loss",
    ],

    default="tie"
)


print("\n")
print("=" * 100)
print("RRF vs BM25 — QUESTION-LEVEL RECALL@10")
print("=" * 100)


print(

    comparison[
        "rrf_vs_bm25_recall10_result"
    ]

    .value_counts()

    .reindex(
        [
            "win",
            "tie",
            "loss",
        ],
        fill_value=0
    )

    .to_string()
)


# ============================================================
# 24. First relevant rank
# ============================================================

print("\n")
print("=" * 100)
print("FIRST RELEVANT RANK")
print("=" * 100)


print(

    hybrid_results[
        "first_relevant_rank"
    ]

    .describe()

    .round(2)

    .to_string()
)


# ============================================================
# 25. Complete
# ============================================================

print("\n")
print("=" * 100)
print("STAGE 4D COMPLETE")
print("=" * 100)


print(
    f"\nHybrid question metrics:\n"
    f"{HYBRID_RESULTS_PATH}"
)

print(
    f"\nHybrid rankings:\n"
    f"{HYBRID_RANKINGS_PATH}"
)

print(
    f"\nHybrid summary:\n"
    f"{HYBRID_SUMMARY_PATH}"
)

print(
    f"\nThree-way comparison:\n"
    f"{HYBRID_COMPARISON_PATH}"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1547.52it/s]




STAGE 4D — HYBRID RRF

RRF k: 60
Evaluation questions: 19


OVERALL METRICS
segment  questions  mean_recall_at_5  mean_recall_at_10  hit_rate_at_5  hit_rate_at_10   mrr
overall         19             0.265              0.476          0.789           0.842 0.582


METRICS BY QUESTION TYPE
    segment  questions  mean_recall_at_5  mean_recall_at_10  hit_rate_at_5  hit_rate_at_10   mrr
 analytical         10             0.111              0.255            0.7             0.8 0.445
comparative          5             0.333              0.558            0.8             0.8 0.657
    factual          4             0.565              0.929            1.0             1.0 0.833


METRICS BY ROUTE
        segment  questions  mean_recall_at_5  mean_recall_at_10  hit_rate_at_5  hit_rate_at_10   mrr
   route:hybrid          9             0.163              0.297          0.778           0.778 0.454
route:retrieval         10             0.357              0.638          0.800           0.900 0.698

In [8]:
# ============================================================
# STAGE 4E — RETRIEVAL ERROR ANALYSIS
# ============================================================

from pathlib import Path
import json
import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

PROJECT_ROOT = Path("..").resolve()

EVAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "retrieval"
)

ERROR_DIR = (
    RESULTS_DIR
    / "error_analysis"
)

ERROR_DIR.mkdir(
    parents=True,
    exist_ok=True
)


EVAL_PATH = (
    EVAL_DIR
    / "evaluation_questions_v2.csv"
)

BM25_RANKINGS_PATH = (
    RESULTS_DIR
    / "bm25_trial_rankings.parquet"
)

DENSE_RANKINGS_PATH = (
    RESULTS_DIR
    / "dense_trial_rankings.parquet"
)

HYBRID_RANKINGS_PATH = (
    RESULTS_DIR
    / "hybrid_rrf_trial_rankings.parquet"
)

HYBRID_METRICS_PATH = (
    RESULTS_DIR
    / "hybrid_rrf_question_metrics.csv"
)


# ============================================================
# 2. Load
# ============================================================

evaluation = pd.read_csv(
    EVAL_PATH
)

bm25 = pd.read_parquet(
    BM25_RANKINGS_PATH
)

dense = pd.read_parquet(
    DENSE_RANKINGS_PATH
)

hybrid = pd.read_parquet(
    HYBRID_RANKINGS_PATH
)

hybrid_metrics = pd.read_csv(
    HYBRID_METRICS_PATH
)


# ============================================================
# 3. Helpers
# ============================================================

def parse_json_list(x):

    if isinstance(x, list):
        return x

    if isinstance(x, str):

        try:
            parsed = json.loads(x)

            if isinstance(parsed, list):
                return parsed

        except Exception:
            pass

    return []


evaluation[
    "gold_ids"
] = (
    evaluation[
        "gold_evidence_nct_ids"
    ]
    .apply(parse_json_list)
)


# ============================================================
# 4. Select retrieval failures
#
# Focus on questions where hybrid retrieval failed badly.
# This is DIAGNOSTIC only.
# ============================================================

TARGET_QUESTIONS = [
    "A07",
    "A06",
    "C03",
]


questions = (
    evaluation.loc[
        evaluation[
            "question_id"
        ]
        .isin(
            TARGET_QUESTIONS
        ),
        [
            "question_id",
            "question_type",
            "route",
            "question",
            "gold_ids",
        ]
    ]
    .copy()
)


# ============================================================
# 5. Gold evidence ranks by retrieval method
# ============================================================

def gold_rank_table(
    qid,
    gold_ids
):

    rows = []


    for method, df in [
        ("BM25", bm25),
        ("Dense", dense),
        ("Hybrid_RRF", hybrid),
    ]:

        q_rankings = (
            df.loc[
                df[
                    "question_id"
                ]
                == qid
            ]
            .copy()
        )


        rank_map = dict(
            zip(
                q_rankings[
                    "nct_id"
                ],
                q_rankings[
                    "trial_rank"
                ],
            )
        )


        for gold_id in gold_ids:

            rank = rank_map.get(
                gold_id
            )


            rows.append({

                "question_id":
                    qid,

                "method":
                    method,

                "gold_nct_id":
                    gold_id,

                "rank_if_top20":
                    rank,

                "retrieved_top5":
                    (
                        rank is not None
                        and
                        rank <= 5
                    ),

                "retrieved_top10":
                    (
                        rank is not None
                        and
                        rank <= 10
                    ),

                "retrieved_top20":
                    (
                        rank is not None
                    ),
            })


    return pd.DataFrame(
        rows
    )


gold_rank_tables = []


for _, q in questions.iterrows():

    gold_rank_tables.append(

        gold_rank_table(
            q[
                "question_id"
            ],
            q[
                "gold_ids"
            ],
        )
    )


gold_ranks = pd.concat(
    gold_rank_tables,
    ignore_index=True
)


# ============================================================
# 6. Top retrieved trials for each method
# ============================================================

def top_results(
    df,
    method,
    qid,
    gold_ids,
    k=10
):

    subset = (
        df.loc[
            df[
                "question_id"
            ]
            == qid
        ]
        .sort_values(
            "trial_rank"
        )
        .head(k)
        .copy()
    )


    subset[
        "method"
    ] = method


    subset[
        "is_gold"
    ] = (
        subset[
            "nct_id"
        ]
        .isin(
            gold_ids
        )
    )


    keep = [
        "question_id",
        "method",
        "trial_rank",
        "nct_id",
        "is_gold",
        "canonical_company",
        "brief_title",
    ]


    # Best chunk information differs slightly by method
    if method == "BM25":

        subset[
            "best_chunk_type"
        ] = subset[
            "best_chunk_type"
        ]

        subset[
            "score"
        ] = subset[
            "bm25_score"
        ]


    elif method == "Dense":

        subset[
            "best_chunk_type"
        ] = subset[
            "best_chunk_type"
        ]

        subset[
            "score"
        ] = subset[
            "dense_score"
        ]


    else:

        subset[
            "best_chunk_type"
        ] = (
            subset[
                "bm25_best_chunk_type"
            ]
            .astype(str)
            +
            " | "
            +
            subset[
                "dense_best_chunk_type"
            ]
            .astype(str)
        )

        subset[
            "score"
        ] = subset[
            "rrf_score"
        ]


    keep += [
        "best_chunk_type",
        "score",
    ]


    return subset[
        keep
    ]


top_tables = []


for _, q in questions.iterrows():

    qid = q[
        "question_id"
    ]

    gold_ids = q[
        "gold_ids"
    ]


    top_tables.extend(
        [
            top_results(
                bm25,
                "BM25",
                qid,
                gold_ids,
            ),

            top_results(
                dense,
                "Dense",
                qid,
                gold_ids,
            ),

            top_results(
                hybrid,
                "Hybrid_RRF",
                qid,
                gold_ids,
            ),
        ]
    )


top10 = pd.concat(
    top_tables,
    ignore_index=True
)


# ============================================================
# 7. Retriever overlap
#
# Helps determine whether fusion failed because both
# retrievers made similar errors or different errors.
# ============================================================

overlap_rows = []


for qid in TARGET_QUESTIONS:

    bm25_top10 = set(
        bm25.loc[
            bm25[
                "question_id"
            ]
            == qid
        ]
        .sort_values(
            "trial_rank"
        )
        .head(10)[
            "nct_id"
        ]
    )


    dense_top10 = set(
        dense.loc[
            dense[
                "question_id"
            ]
            == qid
        ]
        .sort_values(
            "trial_rank"
        )
        .head(10)[
            "nct_id"
        ]
    )


    intersection = (
        bm25_top10
        &
        dense_top10
    )

    union = (
        bm25_top10
        |
        dense_top10
    )


    overlap_rows.append({

        "question_id":
            qid,

        "bm25_dense_top10_overlap":
            len(
                intersection
            ),

        "top10_jaccard":
            (
                len(intersection)
                /
                len(union)
            ),

        "bm25_only":
            len(
                bm25_top10
                -
                dense_top10
            ),

        "dense_only":
            len(
                dense_top10
                -
                bm25_top10
            ),
    })


overlap = pd.DataFrame(
    overlap_rows
)


# ============================================================
# 8. Compact failure summary
# ============================================================

failure_summary = (

    hybrid_metrics.loc[
        hybrid_metrics[
            "question_id"
        ]
        .isin(
            TARGET_QUESTIONS
        ),
        [
            "question_id",
            "question_type",
            "route",
            "gold_count",
            "recall_at_5",
            "recall_at_10",
            "mrr",
            "first_relevant_rank",
            "question",
        ]
    ]

    .merge(
        overlap,
        on="question_id",
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# 9. Diagnostic categories
#
# These are architectural interpretations, not model tuning.
# ============================================================

ERROR_CLASS = {

    "A07":
        (
            "Semantic breadth problem: question asks for "
            "representative evidence across heterogeneous "
            "patient subpopulations/clinical contexts. "
            "Relevant evidence is distributed across many "
            "differently worded trials."
        ),

    "A06":
        (
            "Cross-portfolio synthesis problem: 'strategy' "
            "and relative portfolio scale are not facts stated "
            "inside a single trial document. Structured "
            "portfolio aggregation is required."
        ),

    "C03":
        (
            "Hybrid aggregation problem: development scale "
            "requires structured counts while trial objectives "
            "require narrative evidence. Pure retrieval cannot "
            "fully answer the question."
        ),
}


failure_summary[
    "diagnostic_class"
] = (
    failure_summary[
        "question_id"
    ]
    .map(
        ERROR_CLASS
    )
)


# ============================================================
# 10. Save
# ============================================================

failure_summary.to_csv(
    ERROR_DIR
    / "retrieval_failure_summary.csv",
    index=False
)


gold_ranks.to_csv(
    ERROR_DIR
    / "gold_evidence_rank_analysis.csv",
    index=False
)


top10.to_csv(
    ERROR_DIR
    / "failure_top10_retrievals.csv",
    index=False
)


overlap.to_csv(
    ERROR_DIR
    / "retriever_overlap.csv",
    index=False
)


# ============================================================
# 11. Output
# ============================================================

print("\n")
print("=" * 100)
print("STAGE 4E — RETRIEVAL ERROR ANALYSIS")
print("=" * 100)


print("\n")
print("=" * 100)
print("FAILURE SUMMARY")
print("=" * 100)


print(
    failure_summary[
        [
            "question_id",
            "route",
            "gold_count",
            "recall_at_10",
            "mrr",
            "first_relevant_rank",
            "bm25_dense_top10_overlap",
            "top10_jaccard",
            "diagnostic_class",
        ]
    ]
    .round(3)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 100)
print("GOLD EVIDENCE RETRIEVAL")
print("=" * 100)


gold_summary = (

    gold_ranks

    .groupby(
        [
            "question_id",
            "method",
        ]
    )

    .agg(
        gold_documents=(
            "gold_nct_id",
            "count"
        ),

        gold_in_top5=(
            "retrieved_top5",
            "sum"
        ),

        gold_in_top10=(
            "retrieved_top10",
            "sum"
        ),

        gold_in_top20=(
            "retrieved_top20",
            "sum"
        ),
    )

    .reset_index()
)


print(
    gold_summary
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 100)
print("TOP-10 RETRIEVALS")
print("=" * 100)


for qid in TARGET_QUESTIONS:

    print(
        f"\n{'-' * 100}"
    )

    print(
        qid
    )

    print(
        "-" * 100
    )


    print(

        top10.loc[
            top10[
                "question_id"
            ]
            == qid,
            [
                "method",
                "trial_rank",
                "nct_id",
                "is_gold",
                "canonical_company",
                "best_chunk_type",
                "brief_title",
            ]
        ]

        .to_string(
            index=False
        )
    )


print("\n")
print("=" * 100)
print("STAGE 4E COMPLETE")
print("=" * 100)

print(
    f"\nError-analysis outputs:\n"
    f"{ERROR_DIR}"
)



STAGE 4E — RETRIEVAL ERROR ANALYSIS


FAILURE SUMMARY
question_id     route  gold_count  recall_at_10   mrr  first_relevant_rank  bm25_dense_top10_overlap  top10_jaccard                                                                                                                                                                                           diagnostic_class
        C03    hybrid           6           0.0 0.083                 12.0                         2          0.111                               Hybrid aggregation problem: development scale requires structured counts while trial objectives require narrative evidence. Pure retrieval cannot fully answer the question.
        A06    hybrid           5           0.0 0.053                 19.0                         2          0.111                              Cross-portfolio synthesis problem: 'strategy' and relative portfolio scale are not facts stated inside a single trial document. Structured portfolio aggregation 